In [12]:
import pandas as pd
df = pd.read_parquet(r"C:\Users\HPP\Bowleriq\abt_with_bpi_rowwise.parquet")
print(df.columns.to_list())

['match_id', 'bowler', 'bowling_team', 'opponent_team', 'match_type', 'balls_bowled', 'runs_conceded', 'wickets', 'dot_balls', 'total_extras', 'wides', 'noballs', 'byes', 'legbyes', 'fours_conceded', 'sixes_conceded', 'wickets_bowled', 'wickets_caught', 'wickets_lbw', 'wickets_stumped', 'wickets_caught_and_bowled', 'wickets_hit_wicket', 'balls_powerplay_t20', 'balls_middle_t20', 'balls_death_t20', 'runs_powerplay_t20', 'runs_middle_t20', 'runs_death_t20', 'wickets_powerplay_t20', 'wickets_middle_t20', 'wickets_death_t20', 'dots_powerplay_t20', 'dots_death_t20', 'balls_powerplay_odi', 'balls_middle_odi', 'balls_death_odi', 'runs_powerplay_odi', 'runs_middle_odi', 'runs_death_odi', 'wickets_powerplay_odi', 'wickets_middle_odi', 'wickets_death_odi', 'balls_team_spell_1', 'balls_team_spell_2', 'runs_team_spell_1', 'runs_team_spell_2', 'wickets_team_spell_1', 'wickets_team_spell_2', 'first_over', 'last_over', 'overs_bowled_distinct', 'balls_to_rhb', 'balls_to_lhb', 'economy_rate', 'strike_r

In [6]:
"""
BowlerIQ — Comprehensive Modelling Evaluation Pipeline
=======================================================
Tests 5 target strategies × 2 models × 3 formats
Outputs: result tables, SHAP plots, baseline comparisons
Estimated runtime: 8–15 minutes (LightGBM fast mode)

Approaches tested:
  REG-1  : Regression  → BPI_Final (your original target)
  REG-2  : Regression  → economy_rate (single outcome)
  REG-3  : Regression  → economy deviation from own last-5 avg (relative baseline)
  CLS-1  : Classification → BPI tercile (Top / Mid / Bottom)
  CLS-2  : Classification → outperformed own economy baseline (binary)
"""

# ── Imports ───────────────────────────────────────────────────────────────────
import warnings, os, time
warnings.filterwarnings("ignore")

import numpy  as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # non-interactive — safe for scripts / notebooks
import matplotlib.pyplot as plt

import shap

from sklearn.preprocessing      import LabelEncoder, StandardScaler
from sklearn.impute              import SimpleImputer
from sklearn.linear_model       import Ridge, LogisticRegression
from sklearn.metrics            import (
    r2_score, mean_absolute_error, mean_squared_error,
    f1_score, matthews_corrcoef, roc_auc_score,
    accuracy_score, classification_report,
)
import lightgbm as lgb

# ── Config ────────────────────────────────────────────────────────────────────
DATA_PATH  = r"C:\Users\HPP\Bowleriq\abt_with_bpi_rowwise.parquet"
OUTPUT_DIR = r"C:\Users\HPP\Bowleriq\modelling_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FORMATS    = ["T20", "ODI", "Test"]
RAND_STATE = 42

# ── Column taxonomy ───────────────────────────────────────────────────────────

# Columns we read as targets BEFORE dropping leakage
TARGET_RAW = [
    "economy_rate", "wickets", "strike_rate", "dot_ball_percentage",
    "BPI_Final", "economy_last_5",
]

# ── Same-match bowling outcomes (leakage) ──
LEAKAGE_OUTCOMES = [
    "balls_bowled","runs_conceded","wickets","dot_balls","total_extras",
    "wides","noballs","byes","legbyes","fours_conceded","sixes_conceded",
    "wickets_bowled","wickets_caught","wickets_lbw","wickets_stumped",
    "wickets_caught_and_bowled","wickets_hit_wicket",
    "balls_powerplay_t20","balls_middle_t20","balls_death_t20",
    "runs_powerplay_t20","runs_middle_t20","runs_death_t20",
    "wickets_powerplay_t20","wickets_middle_t20","wickets_death_t20",
    "dots_powerplay_t20","dots_death_t20",
    "balls_powerplay_odi","balls_middle_odi","balls_death_odi",
    "runs_powerplay_odi","runs_middle_odi","runs_death_odi",
    "wickets_powerplay_odi","wickets_middle_odi","wickets_death_odi",
    "balls_team_spell_1","balls_team_spell_2",
    "runs_team_spell_1","runs_team_spell_2",
    "wickets_team_spell_1","wickets_team_spell_2",
    "first_over","last_over","overs_bowled_distinct",
    "balls_to_rhb","balls_to_lhb",
    "economy_rate","strike_rate","bowling_average",
    "dot_ball_percentage","overs_bowled",
    "boundary_percentage","bowled_percentage","caught_percentage",
    "lbw_percentage","rhb_percentage","lhb_percentage",
    "dots_powerplay_pct_t20","dots_death_pct_t20",
    # Current-match result columns
    "winner","result_type","win_by_runs","win_by_wickets","win_method",
    "bowling_team_won",
]

# ── BPI components (leakage) ──
LEAKAGE_BPI = [
    "economy_score","strike_score","wicket_score","dot_score",
    "BPI_Raw","Opposition_Factor","Opposition_Factor_Capped",
    "BPI_OppAdj_Capped","BPI_Final",
]

# ── Pure identifiers / metadata not useful as features ──
IDENTIFIERS = [
    "match_id","bowler","bowler_espn_id","bowler_full_name","bowler_dob",
    "venue_id","venue_original","venue","venue_canonical","venue_city",
    "bowling_team","opponent_team",
    "bowling_team_country","bowling_team_code","bowling_team_region",
    "opponent_team_country","opponent_team_code","opponent_team_region",
    "toss_winner",    # raw team name string — too sparse, no predictive value
    "match_type",     # used for format split only
    "date",           # used for temporal split only
]

ALL_DROP = set(LEAKAGE_OUTCOMES) | set(LEAKAGE_BPI) | set(IDENTIFIERS)

# ── String categoricals to label-encode ──
CAT_COLS = [
    "bowler_batting_style","bowler_bowling_style","toss_decision",
    "career_stage","bowling_team_type","bowling_team_member_type",
    "opponent_team_type","opponent_team_member_type",
    "bowler_metadata_quality","venue_country",
]

# ── Bool-like columns → cast to int 0/1 ──
BOOL_COLS = [
    "is_home_match","bowling_team_won_toss","is_debut","economy_improving",
    "venue_specialist","opponent_specialist","first_time_at_venue",
    "first_time_vs_opponent","opponent_is_full_member","opponent_is_top8",
]

# ── Format-exclusive feature sets ──
FORMAT_EXCLUSIVE = {
    "T20" : ["career_matches_t20","career_wickets_t20","career_economy_t20",
             "wickets_last_5_t20","economy_last_5_t20"],
    "ODI" : ["career_matches_odi","career_wickets_odi","career_economy_odi",
             "wickets_last_5_odi","economy_last_5_odi"],
    "Test": ["career_matches_test","career_wickets_test","career_economy_test",
             "wickets_last_5_test","economy_last_5_test"],
}
FORMAT_CROSS_DROP = {
    "T20" : set(FORMAT_EXCLUSIVE["ODI"])  | set(FORMAT_EXCLUSIVE["Test"]),
    "ODI" : set(FORMAT_EXCLUSIVE["T20"])  | set(FORMAT_EXCLUSIVE["Test"]),
    "Test": set(FORMAT_EXCLUSIVE["T20"])  | set(FORMAT_EXCLUSIVE["ODI"]),
}

# ── LightGBM params ──
LGB_BASE = dict(
    n_estimators=300, learning_rate=0.05, num_leaves=31,
    min_child_samples=30, subsample=0.8, colsample_bytree=0.8,
    random_state=RAND_STATE, n_jobs=-1, verbosity=-1,
)

# ══════════════════════════════════════════════════════════════════════════════
# Utility functions
# ══════════════════════════════════════════════════════════════════════════════

def encode_and_cast(df):
    """
    Label-encode all string categoricals and cast booleans → int.
    Works on a copy; does NOT rely on remaining object columns making
    it through to the scaler (that was the source of the original error).
    """
    for col in CAT_COLS:
        if col in df.columns:
            df[col] = df[col].fillna("__MISSING__").astype(str)
            df[col] = LabelEncoder().fit_transform(df[col])
    for col in BOOL_COLS:
        if col in df.columns:
            df[col] = df[col].fillna(0)
            # Handle True/False strings, bool dtype, or already numeric
            df[col] = df[col].map(
                lambda x: 1 if str(x).strip().lower() in ("1","true","yes") else 0
            ).astype(int)
    return df


def build_feature_matrix(df, fmt):
    """
    Drop leakage + identifier + cross-format columns.
    Impute remaining numeric columns.
    Returns (X DataFrame, feature_col_names list).
    """
    drop_here = ALL_DROP | FORMAT_CROSS_DROP[fmt] | {"economy_last_5"}
    feat_cols = []
    for c in df.columns:
        if c in drop_here:
            continue
        if df[c].dtype == object:
            # Should not happen after encode_and_cast, but guard anyway
            continue
        feat_cols.append(c)

    X = df[feat_cols].copy()
    imp = SimpleImputer(strategy="median")
    X_arr = imp.fit_transform(X)
    X = pd.DataFrame(X_arr, columns=feat_cols, index=df.index)
    return X, feat_cols


def reg_metrics(y_true, y_pred, y_naive):
    r2   = r2_score(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    bm   = mean_absolute_error(y_true, y_naive)
    uplift = (bm - mae) / bm * 100 if bm > 0 else 0.0
    return dict(R2=round(r2,4), MAE=round(mae,4), RMSE=round(rmse,4),
                Naive_MAE=round(bm,4), Uplift_pct=round(uplift,2))


def cls_metrics(y_true, y_pred, y_prob=None):
    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average="macro", zero_division=0)
    mcc = matthews_corrcoef(y_true, y_pred)
    auc = "N/A"
    if y_prob is not None:
        try:
            nc = y_prob.shape[1]
            if nc == 2:
                auc = round(roc_auc_score(y_true, y_prob[:,1]), 4)
            else:
                auc = round(roc_auc_score(y_true, y_prob,
                                          multi_class="ovr", average="macro"), 4)
        except Exception:
            pass
    return dict(Accuracy=round(acc,4), F1_macro=round(f1,4),
                MCC=round(mcc,4), AUC=auc)


def save_shap(model, X_te, feat_cols, fmt, tag):
    try:
        X_s = X_te.sample(min(500, len(X_te)), random_state=RAND_STATE)
        exp = shap.TreeExplainer(model)
        sv  = exp.shap_values(X_s)
        if isinstance(sv, list):          # multiclass
            sv = np.mean([np.abs(a) for a in sv], axis=0)
        mean_abs = np.abs(sv).mean(axis=0)
        idx  = np.argsort(mean_abs)[-20:][::-1]
        fig, ax = plt.subplots(figsize=(8,6))
        ax.barh([feat_cols[i] for i in idx[::-1]],
                mean_abs[idx[::-1]], color="steelblue")
        ax.set_xlabel("Mean |SHAP|")
        ax.set_title(f"Feature Importance — {fmt} / {tag}")
        plt.tight_layout()
        fpath = os.path.join(OUTPUT_DIR, f"shap_{fmt}_{tag}.png")
        plt.savefig(fpath, dpi=120);  plt.close()
        print(f"      SHAP → {fpath}")
    except Exception as e:
        print(f"      SHAP skipped: {e}")


# ══════════════════════════════════════════════════════════════════════════════
# Per-format pipeline
# ══════════════════════════════════════════════════════════════════════════════

def run_format(fmt, df_fmt):
    print(f"\n{'='*60}")
    print(f"  FORMAT: {fmt}   ({len(df_fmt):,} rows)")
    print(f"{'='*60}")

    # 1. Encode strings / bools
    df_fmt = encode_and_cast(df_fmt)

    # 2. Snapshot raw targets (before we drop leakage)
    T = {}
    for col in TARGET_RAW:
        T[col] = df_fmt[col].copy() if col in df_fmt.columns else pd.Series(dtype=float)

    T["eco_dev"]      = T["economy_rate"] - T["economy_last_5"]
    T["outperformed"] = (T["eco_dev"] < 0).astype(int)   # 1 = bowler beat own average

    # BPI tercile (0=Bottom, 1=Mid, 2=Top)
    valid_bpi = T["BPI_Final"].dropna()
    bpi_q = pd.qcut(valid_bpi, q=3, labels=False, duplicates="drop")
    T["bpi_tercile"] = bpi_q.reindex(df_fmt.index).fillna(1).astype(int)

    # 3. Build feature matrix
    X_all, feat_cols = build_feature_matrix(df_fmt, fmt)
    print(f"  Features used: {len(feat_cols)}")

    # 4. Temporal split (80/20 chronological)
    dates = df_fmt["date"]
    sorted_idx = dates.sort_values().index
    n_tr = int(len(sorted_idx) * 0.80)
    tr_idx = sorted_idx[:n_tr]
    te_idx = sorted_idx[n_tr:]
    print(f"  Train: {len(tr_idx):,}   Test: {len(te_idx):,}")

    X_tr, X_te = X_all.loc[tr_idx], X_all.loc[te_idx]

    scaler  = StandardScaler()
    X_tr_s  = scaler.fit_transform(X_tr)
    X_te_s  = scaler.transform(X_te)

    reg_rows, cls_rows = [], []

    # ── REGRESSION ────────────────────────────────────────────────────────────
    REG_MAP = {
        "REG1_BPI"     : "BPI_Final",
        "REG2_Economy" : "economy_rate",
        "REG3_EcoDev"  : "eco_dev",
    }
    for tag, tkey in REG_MAP.items():
        y_all = T[tkey]
        y_tr  = y_all.loc[tr_idx]
        y_te  = y_all.loc[te_idx]
        # Naive baseline: predict using bowler's own last-5 economy
        naive = T["economy_last_5"].loc[te_idx]

        # Drop rows where target is NaN
        mask_tr = ~y_tr.isna()
        mask_te = ~y_te.isna()

        print(f"\n  ── {tag} ──  (target NaN train:{(~mask_tr).sum()}  test:{(~mask_te).sum()})")

        # Ridge
        r = Ridge(alpha=1.0)
        r.fit(X_tr_s[mask_tr], y_tr[mask_tr])
        yp_r = r.predict(X_te_s[mask_te])
        m = reg_metrics(y_te[mask_te], yp_r, naive[mask_te])
        reg_rows.append({"Format":fmt,"Approach":tag,"Model":"Ridge",**m})
        print(f"    Ridge    R²={m['R2']:+.4f}  MAE={m['MAE']:.4f}  Uplift={m['Uplift_pct']:.1f}%")

        # LightGBM
        lg = lgb.LGBMRegressor(**LGB_BASE)
        lg.fit(
            X_tr.loc[mask_tr], y_tr[mask_tr],
            eval_set=[(X_te.loc[mask_te], y_te[mask_te])],
            callbacks=[lgb.early_stopping(30,verbose=False), lgb.log_evaluation(-1)],
        )
        yp_l = lg.predict(X_te.loc[mask_te])
        m = reg_metrics(y_te[mask_te], yp_l, naive[mask_te])
        reg_rows.append({"Format":fmt,"Approach":tag,"Model":"LightGBM",**m})
        print(f"    LightGBM R²={m['R2']:+.4f}  MAE={m['MAE']:.4f}  Uplift={m['Uplift_pct']:.1f}%")

        save_shap(lg, X_te.loc[mask_te], feat_cols, fmt, tag)

    # ── CLASSIFICATION ────────────────────────────────────────────────────────
    CLS_MAP = {
        "CLS1_BPITercile" : "bpi_tercile",
        "CLS2_OutvBase"   : "outperformed",
    }
    for tag, tkey in CLS_MAP.items():
        y_all = T[tkey]
        y_tr  = y_all.loc[tr_idx]
        y_te  = y_all.loc[te_idx]

        mask_tr = ~y_tr.isna()
        mask_te = ~y_te.isna()
        y_tr_c = y_tr[mask_tr].astype(int)
        y_te_c = y_te[mask_te].astype(int)

        print(f"\n  ── {tag} ──")
        print(f"    Train class dist: {dict(y_tr_c.value_counts().sort_index())}")
        print(f"    Test  class dist: {dict(y_te_c.value_counts().sort_index())}")

        # Logistic Regression
        lr = LogisticRegression(max_iter=1000, C=1.0, class_weight="balanced",
                                 random_state=RAND_STATE, n_jobs=-1)
        lr.fit(X_tr_s[mask_tr], y_tr_c)
        yp_lr = lr.predict(X_te_s[mask_te])
        ypr   = lr.predict_proba(X_te_s[mask_te])
        m = cls_metrics(y_te_c, yp_lr, ypr)
        cls_rows.append({"Format":fmt,"Approach":tag,"Model":"LogReg",**m})
        print(f"    LogReg   Acc={m['Accuracy']:.4f}  F1={m['F1_macro']:.4f}  MCC={m['MCC']:.4f}  AUC={m['AUC']}")

        # LightGBM
        lgp = {**LGB_BASE, "class_weight":"balanced"}
        lc = lgb.LGBMClassifier(**lgp)
        lc.fit(
            X_tr.loc[mask_tr], y_tr_c,
            eval_set=[(X_te.loc[mask_te], y_te_c)],
            callbacks=[lgb.early_stopping(30,verbose=False), lgb.log_evaluation(-1)],
        )
        yp_lc  = lc.predict(X_te.loc[mask_te])
        ypr_lc = lc.predict_proba(X_te.loc[mask_te])
        m = cls_metrics(y_te_c, yp_lc, ypr_lc)
        cls_rows.append({"Format":fmt,"Approach":tag,"Model":"LightGBM",**m})
        print(f"    LightGBM Acc={m['Accuracy']:.4f}  F1={m['F1_macro']:.4f}  MCC={m['MCC']:.4f}  AUC={m['AUC']}")

        # Detailed report
        n_cls = y_te_c.nunique()
        if n_cls == 3:
            lnames = ["Bottom","Mid","Top"]
        else:
            lnames = ["Below Avg","Above Avg"]
        target_names = lnames[:n_cls]
        print(f"\n    Classification Report (LightGBM):")
        print(classification_report(y_te_c, yp_lc,
                                    target_names=target_names, zero_division=0))

        save_shap(lc, X_te.loc[mask_te], feat_cols, fmt, tag)

    return pd.DataFrame(reg_rows), pd.DataFrame(cls_rows)


# ══════════════════════════════════════════════════════════════════════════════
# Summary + interpretation
# ══════════════════════════════════════════════════════════════════════════════

def print_summary(reg_df, cls_df):
    bar = "═" * 68

    print(f"\n\n{bar}")
    print("  REGRESSION RESULTS (all formats)")
    print(bar)
    print(reg_df.to_string(index=False))

    print(f"\n  ── Best model per Format × Target ──")
    best_r = reg_df.loc[reg_df.groupby(["Format","Approach"])["R2"].idxmax()]
    print(best_r[["Format","Approach","Model","R2","MAE","Uplift_pct"]].to_string(index=False))

    print(f"\n\n{bar}")
    print("  CLASSIFICATION RESULTS (all formats)")
    print(bar)
    print(cls_df.to_string(index=False))

    print(f"\n  ── Best model per Format × Target ──")
    best_c = cls_df.loc[cls_df.groupby(["Format","Approach"])["F1_macro"].idxmax()]
    print(best_c[["Format","Approach","Model","Accuracy","F1_macro","MCC","AUC"]].to_string(index=False))

    print(f"""
{bar}
  HOW TO READ THESE RESULTS
{bar}

  REGRESSION
  ┌──────────────────────────────────────────────────────────────┐
  │ R²  < 0.10    Very low signal — target needs rethinking      │
  │ R²  0.10-0.20 Modest signal  — publishable with framing      │
  │ R²  > 0.20    Good for cricket/sports domain                 │
  │ Uplift > 0%   Model beats naive "use last-5 career avg"      │
  │ Uplift < 0%   Naive baseline outperforms your model (!)      │
  └──────────────────────────────────────────────────────────────┘

  CLASSIFICATION
  ┌──────────────────────────────────────────────────────────────┐
  │ F1-macro > 0.40   Acceptable (random 3-class baseline=0.33)  │
  │ MCC      > 0.15   Genuine predictive signal above random     │
  │ MCC      > 0.30   Strong result for binary sports outcome    │
  │ AUC      > 0.60   Model meaningfully separates classes       │
  └──────────────────────────────────────────────────────────────┘

  PAPER ANGLES BASED ON RESULTS
  • REG-3 (EcoDev) > REG-2 (Economy) in R²
      → Relative-to-baseline target works; publish "consistency" angle
  • CLS-2 (OutvBase) has best MCC
      → Binary outperformance framing is your cleanest story
  • Uplift_pct positive across formats
      → Model adds measurable value over career stats alone
  • SHAP plots saved in output dir
      → Use feature group analysis (career vs form vs venue) as a
         separate finding section in the paper

  SHAP output files:
    shap_<FORMAT>_<APPROACH>.png  in  {OUTPUT_DIR}
""")


# ══════════════════════════════════════════════════════════════════════════════
# Entry point
# ══════════════════════════════════════════════════════════════════════════════

def main():
    t0 = time.time()
    print("BowlerIQ Modelling Pipeline — starting")
    print(f"Data: {DATA_PATH}")

    df = pd.read_parquet(DATA_PATH)
    print(f"Loaded {df.shape[0]:,} rows × {df.shape[1]} columns")

    # Normalise match_type
    df["match_type"] = df["match_type"].str.strip().str.upper()
    df.loc[df["match_type"] == "T20I", "match_type"] = "T20"
    print(f"match_type values: {sorted(df['match_type'].unique())}")

    # Ensure date column is datetime
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])
    print(f"After date filter: {len(df):,} rows")

    all_reg, all_cls = [], []

    for fmt in FORMATS:
        mask   = df["match_type"] == fmt.upper()
        df_fmt = df[mask].copy()
        if len(df_fmt) < 300:
            print(f"\nSkipping {fmt} — only {len(df_fmt)} rows")
            continue
        rdf, cdf = run_format(fmt, df_fmt)
        all_reg.append(rdf)
        all_cls.append(cdf)

    if not all_reg:
        print("\nNo formats processed. Unique match_type values found:")
        print(df["match_type"].value_counts())
        return

    reg_df = pd.concat(all_reg, ignore_index=True)
    cls_df = pd.concat(all_cls, ignore_index=True)

    reg_df.to_csv(os.path.join(OUTPUT_DIR, "regression_results.csv"),  index=False)
    cls_df.to_csv(os.path.join(OUTPUT_DIR, "classification_results.csv"), index=False)
    print(f"\nCSVs saved to {OUTPUT_DIR}")

    print_summary(reg_df, cls_df)
    print(f"\nTotal runtime: {(time.time()-t0)/60:.1f} min")


if __name__ == "__main__":
    main()

BowlerIQ Modelling Pipeline — starting
Data: C:\Users\HPP\Bowleriq\abt_with_bpi_rowwise.parquet
Loaded 49,945 rows × 170 columns
match_type values: ['ODI', 'T20', 'TEST']
After date filter: 49,945 rows

  FORMAT: T20   (22,917 rows)
  Features used: 62
  Train: 18,333   Test: 4,584

  ── REG1_BPI ──  (target NaN train:0  test:0)
    Ridge    R²=+0.0533  MAE=13.1270  Uplift=63.6%
    LightGBM R²=+0.0645  MAE=13.0875  Uplift=63.7%
      SHAP → C:\Users\HPP\Bowleriq\modelling_results\shap_T20_REG1_BPI.png

  ── REG2_Economy ──  (target NaN train:0  test:0)
    Ridge    R²=+0.1473  MAE=1.9233  Uplift=20.6%
    LightGBM R²=+0.1561  MAE=1.9209  Uplift=20.7%
      SHAP → C:\Users\HPP\Bowleriq\modelling_results\shap_T20_REG2_Economy.png

  ── REG3_EcoDev ──  (target NaN train:1  test:0)
    Ridge    R²=+0.3156  MAE=1.9726  Uplift=73.9%
    LightGBM R²=+0.3217  MAE=1.9562  Uplift=74.1%
      SHAP → C:\Users\HPP\Bowleriq\modelling_results\shap_T20_REG3_EcoDev.png

  ── CLS1_BPITercile ──
    Tra

In [10]:
"""
BowlerIQ — Refined Modelling Pipeline (EcoDev + BPIDev)
=========================================================
4 target variants across 3 formats × 2 models
Estimated runtime: ~5–8 minutes

Targets:
  eco_dev_form    = economy_rate  -  last-5 economy (same format, fallback→career avg)
  eco_dev_career  = economy_rate  -  career_economy
  bpi_dev_form    = BPI_Final     -  last-5 BPI (same format, rolling, fallback→career BPI avg)
  bpi_dev_career  = BPI_Final     -  career BPI rolling avg (shift-1, expanding)

Key design decisions:
  • Format-specific last-5 uses existing columns: economy_last_5_t20/odi/test
  • If those are NaN (< 5 format matches played), falls back to career_economy
  • BPI rolling last-5 is computed here (doesn't exist in raw data):
      - Sorted by date, grouped by bowler+format
      - shift(1) rolling mean window=5, min_periods=1  → no leakage, no NaN drop
      - Fallback to expanding career BPI mean if still NaN
  • Temporal 80/20 split on date (no random leakage)
  • Models: Ridge (linear baseline) + LightGBM (nonlinear)
  • SHAP: top-20 features saved per format × target
"""

import warnings, os, time
warnings.filterwarnings("ignore")

import numpy  as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import shap

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute         import SimpleImputer
from sklearn.linear_model  import Ridge
from sklearn.metrics        import r2_score, mean_absolute_error, mean_squared_error
import lightgbm as lgb

# ── Config ────────────────────────────────────────────────────────────────────
DATA_PATH  = r"C:\Users\HPP\Bowleriq\abt_with_bpi_rowwise.parquet"
OUTPUT_DIR = r"C:\Users\HPP\Bowleriq\modelling_results_v2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FORMATS    = ["T20", "ODI", "Test"]
RAND_STATE = 42

# ── Column taxonomy ───────────────────────────────────────────────────────────
# Same-match outcome leakage
LEAKAGE_OUTCOMES = [
    "balls_bowled","runs_conceded","wickets","dot_balls","total_extras",
    "wides","noballs","byes","legbyes","fours_conceded","sixes_conceded",
    "wickets_bowled","wickets_caught","wickets_lbw","wickets_stumped",
    "wickets_caught_and_bowled","wickets_hit_wicket",
    "balls_powerplay_t20","balls_middle_t20","balls_death_t20",
    "runs_powerplay_t20","runs_middle_t20","runs_death_t20",
    "wickets_powerplay_t20","wickets_middle_t20","wickets_death_t20",
    "dots_powerplay_t20","dots_death_t20",
    "balls_powerplay_odi","balls_middle_odi","balls_death_odi",
    "runs_powerplay_odi","runs_middle_odi","runs_death_odi",
    "wickets_powerplay_odi","wickets_middle_odi","wickets_death_odi",
    "balls_team_spell_1","balls_team_spell_2",
    "runs_team_spell_1","runs_team_spell_2",
    "wickets_team_spell_1","wickets_team_spell_2",
    "first_over","last_over","overs_bowled_distinct",
    "balls_to_rhb","balls_to_lhb",
    "economy_rate","strike_rate","bowling_average",
    "dot_ball_percentage","overs_bowled",
    "boundary_percentage","bowled_percentage","caught_percentage",
    "lbw_percentage","rhb_percentage","lhb_percentage",
    "dots_powerplay_pct_t20","dots_death_pct_t20",
    "winner","result_type","win_by_runs","win_by_wickets","win_method",
    "bowling_team_won",
]

# BPI components leakage
LEAKAGE_BPI = [
    "economy_score","strike_score","wicket_score","dot_score",
    "BPI_Raw","Opposition_Factor","Opposition_Factor_Capped",
    "BPI_OppAdj_Capped","BPI_Final",
]

# Pure identifiers
IDENTIFIERS = [
    "match_id","bowler","bowler_espn_id","bowler_full_name","bowler_dob",
    "venue_id","venue_original","venue","venue_canonical","venue_city",
    "bowling_team","opponent_team",
    "bowling_team_country","bowling_team_code","bowling_team_region",
    "opponent_team_country","opponent_team_code","opponent_team_region",
    "toss_winner","match_type","date",
    # computed target-construction helpers — not features
    "economy_last_5",
    "economy_last_5_t20","economy_last_5_odi","economy_last_5_test",
    "_bpi_last5_form","_bpi_career_avg",   # engineered cols added below
]

ALL_DROP = set(LEAKAGE_OUTCOMES) | set(LEAKAGE_BPI) | set(IDENTIFIERS)

# Categoricals to label-encode
CAT_COLS = [
    "bowler_batting_style","bowler_bowling_style","toss_decision",
    "career_stage","bowling_team_type","bowling_team_member_type",
    "opponent_team_type","opponent_team_member_type",
    "bowler_metadata_quality","venue_country",
]

# Booleans to cast 0/1
BOOL_COLS = [
    "is_home_match","bowling_team_won_toss","is_debut","economy_improving",
    "venue_specialist","opponent_specialist","first_time_at_venue",
    "first_time_vs_opponent","opponent_is_full_member","opponent_is_top8",
]

# Format-exclusive feature sets (cross-dropped per format model)
FORMAT_EXCLUSIVE = {
    "T20" : ["career_matches_t20","career_wickets_t20","career_economy_t20",
             "wickets_last_5_t20","economy_last_5_t20"],
    "ODI" : ["career_matches_odi","career_wickets_odi","career_economy_odi",
             "wickets_last_5_odi","economy_last_5_odi"],
    "Test": ["career_matches_test","career_wickets_test","career_economy_test",
             "wickets_last_5_test","economy_last_5_test"],
}
FORMAT_CROSS_DROP = {
    "T20" : set(FORMAT_EXCLUSIVE["ODI"])  | set(FORMAT_EXCLUSIVE["Test"]),
    "ODI" : set(FORMAT_EXCLUSIVE["T20"])  | set(FORMAT_EXCLUSIVE["Test"]),
    "Test": set(FORMAT_EXCLUSIVE["T20"])  | set(FORMAT_EXCLUSIVE["ODI"]),
}

# LightGBM params
LGB_PARAMS = dict(
    n_estimators=400, learning_rate=0.04, num_leaves=31,
    min_child_samples=30, subsample=0.8, colsample_bytree=0.8,
    random_state=RAND_STATE, n_jobs=-1, verbosity=-1,
)

# ══════════════════════════════════════════════════════════════════════════════
# Step 1 — Engineer BPI rolling columns (done ONCE on full dataset)
# ══════════════════════════════════════════════════════════════════════════════

def engineer_bpi_rolling(df):
    """
    Computes two columns used for BPI deviation targets:

    _bpi_last5_form   — rolling mean of last-5 BPI_Final values for the
                        same bowler × same format, shifted by 1 to avoid
                        leakage. Uses min_periods=1 so early career rows
                        use whatever history exists (1–4 matches).
                        Falls back to _bpi_career_avg if still NaN.

    _bpi_career_avg   — expanding (all prior matches) mean of BPI_Final
                        per bowler, shift-1. Represents "expected BPI
                        based on full career to date".

    Both are computed on data sorted chronologically.
    """
    print("  Engineering BPI rolling columns...")
    df = df.sort_values(["bowler", "match_type", "date"]).reset_index(drop=True)

    # ── Career BPI avg (cross-format expanding mean, shift-1) ──────────────
    df["_bpi_career_avg"] = (
        df.groupby("bowler")["BPI_Final"]
          .transform(lambda s: s.shift(1).expanding().mean())
    )

    # ── Format-specific last-5 BPI (shift-1, rolling-5, min_periods=1) ──────
    df["_bpi_last5_form"] = (
        df.groupby(["bowler", "match_type"])["BPI_Final"]
          .transform(lambda s: s.shift(1).rolling(window=5, min_periods=1).mean())
    )

    # Fallback: if _bpi_last5_form is NaN (debut match in format),
    # use _bpi_career_avg; if that's also NaN (absolute debut), use
    # the global format median (filled after full computation).
    df["_bpi_last5_form"] = df["_bpi_last5_form"].fillna(df["_bpi_career_avg"])

    # For absolute debuts (NaN in both), fill with format-level median later
    # (done per-format slice below)

    n_missing_last5   = df["_bpi_last5_form"].isna().sum()
    n_missing_career  = df["_bpi_career_avg"].isna().sum()
    print(f"    _bpi_last5_form  NaN remaining: {n_missing_last5}")
    print(f"    _bpi_career_avg  NaN remaining: {n_missing_career}")

    return df


# ══════════════════════════════════════════════════════════════════════════════
# Step 2 — Encoding helpers
# ══════════════════════════════════════════════════════════════════════════════

def encode_and_cast(df):
    """Label-encode categorical strings; cast booleans to int."""
    for col in CAT_COLS:
        if col in df.columns:
            df[col] = df[col].fillna("__MISSING__").astype(str)
            df[col] = LabelEncoder().fit_transform(df[col])
    for col in BOOL_COLS:
        if col in df.columns:
            df[col] = df[col].fillna(0).map(
                lambda x: 1 if str(x).strip().lower() in ("1", "true", "yes") else 0
            ).astype(int)
    return df


# ══════════════════════════════════════════════════════════════════════════════
# Step 3 — Feature matrix builder
# ══════════════════════════════════════════════════════════════════════════════

def build_features(df, fmt):
    """
    Drop leakage + identifiers + cross-format columns + engineered helpers.
    Impute medians. Return (X DataFrame, feature_names list).
    """
    drop_here = ALL_DROP | FORMAT_CROSS_DROP[fmt]
    feat_cols = [
        c for c in df.columns
        if c not in drop_here and df[c].dtype != object
    ]
    X = df[feat_cols].copy()
    imp = SimpleImputer(strategy="median")
    X_arr = imp.fit_transform(X)
    return pd.DataFrame(X_arr, columns=feat_cols, index=df.index), feat_cols


# ══════════════════════════════════════════════════════════════════════════════
# Step 4 — Metrics
# ══════════════════════════════════════════════════════════════════════════════

def reg_metrics(y_true, y_pred, y_naive, naive_label):
    r2      = r2_score(y_true, y_pred)
    mae     = mean_absolute_error(y_true, y_pred)
    rmse    = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    naive_m = mean_absolute_error(y_true, y_naive)
    uplift  = (naive_m - mae) / naive_m * 100 if naive_m > 0 else 0.0
    return dict(
        R2        = round(r2,    4),
        MAE       = round(mae,   4),
        RMSE      = round(rmse,  4),
        Naive_MAE = round(naive_m, 4),
        Naive_src = naive_label,
        Uplift_pct= round(uplift, 2),
    )


# ══════════════════════════════════════════════════════════════════════════════
# Step 5 — SHAP plot
# ══════════════════════════════════════════════════════════════════════════════

def save_shap(model, X_te, feat_cols, fmt, tag, out_dir):
    try:
        X_s  = X_te.sample(min(600, len(X_te)), random_state=RAND_STATE)
        exp  = shap.TreeExplainer(model)
        sv   = exp.shap_values(X_s)

        # sv is ndarray for regression (n_samples × n_features)
        if not isinstance(sv, np.ndarray):
            sv = np.array(sv)
        if sv.ndim == 3:          # multiclass fallback
            sv = np.mean(np.abs(sv), axis=0)

        mean_abs = np.abs(sv).mean(axis=0)

        # Safety: lengths must match
        n = min(len(feat_cols), len(mean_abs))
        cols_   = feat_cols[:n]
        vals_   = mean_abs[:n]

        idx = np.argsort(vals_)[-20:][::-1]

        fig, ax = plt.subplots(figsize=(9, 6))
        ax.barh([cols_[i] for i in idx[::-1]],
                [vals_[i] for i in idx[::-1]],
                color="steelblue")
        ax.set_xlabel("Mean |SHAP value|")
        ax.set_title(f"Top-20 Features — {fmt} / {tag}")
        plt.tight_layout()
        fpath = os.path.join(out_dir, f"shap_{fmt}_{tag}.png")
        plt.savefig(fpath, dpi=120)
        plt.close()
        print(f"      SHAP → {fpath}")
    except Exception as e:
        print(f"      SHAP skipped ({e})")


# ══════════════════════════════════════════════════════════════════════════════
# Step 6 — Per-format pipeline
# ══════════════════════════════════════════════════════════════════════════════

def run_format(fmt, df_fmt):
    print(f"\n{'='*60}")
    print(f"  FORMAT: {fmt}   ({len(df_fmt):,} rows)")
    print(f"{'='*60}")

    # Map format string to the right existing column names
    fmt_lower = fmt.lower()
    eco_last5_col  = f"economy_last_5_{fmt_lower}"   # e.g. economy_last_5_t20

    # ── Encode strings / booleans ─────────────────────────────────────────────
    df_fmt = encode_and_cast(df_fmt)

    # ── Build the 4 targets ───────────────────────────────────────────────────
    eco_raw     = df_fmt["economy_rate"].copy()
    bpi_raw     = df_fmt["BPI_Final"].copy()
    career_eco  = df_fmt["career_economy"].copy()

    # Format-level medians for absolute-debut fallback
    bpi_last5_median   = df_fmt["_bpi_last5_form"].median()
    bpi_career_median  = df_fmt["_bpi_career_avg"].median()
    eco_career_median  = career_eco.median()

    # ── eco_dev_form ──────────────────────────────────────────────────────────
    # Use format-specific last-5 economy column
    # Fallback chain: eco_last5_format → career_economy → format median
    if eco_last5_col in df_fmt.columns:
        eco_baseline_form = df_fmt[eco_last5_col].copy()
    else:
        print(f"  WARNING: {eco_last5_col} not found — using career_economy as form baseline")
        eco_baseline_form = career_eco.copy()

    eco_baseline_form = (eco_baseline_form
                         .fillna(career_eco)           # < 5 format matches → use career avg
                         .fillna(eco_career_median))   # absolute debut → format median

    eco_dev_form   = eco_raw - eco_baseline_form
    eco_dev_career = eco_raw - career_eco.fillna(eco_career_median)

    # ── bpi_dev_form ──────────────────────────────────────────────────────────
    bpi_baseline_form = (df_fmt["_bpi_last5_form"]
                         .fillna(bpi_last5_median))    # any remaining NaN → median

    bpi_baseline_career = (df_fmt["_bpi_career_avg"]
                           .fillna(bpi_career_median))

    bpi_dev_form   = bpi_raw - bpi_baseline_form
    bpi_dev_career = bpi_raw - bpi_baseline_career

    # Summary of fallback usage
    n_form_fallback = eco_baseline_form.isna().sum()   # should be 0 after fills
    n_bpi_debut     = df_fmt["_bpi_last5_form"].isna().sum()
    print(f"  Eco baseline (form):   {eco_last5_col}  |  Career fallback used: "
          f"{df_fmt[eco_last5_col].isna().sum() if eco_last5_col in df_fmt.columns else 'N/A'} rows")
    print(f"  BPI last-5 debut rows (→ career fallback used): {n_bpi_debut}")

    # ── Feature matrix ────────────────────────────────────────────────────────
    X_all, feat_cols = build_features(df_fmt, fmt)
    print(f"  Features used: {len(feat_cols)}")

    # ── Temporal split (80/20) ────────────────────────────────────────────────
    sorted_idx = df_fmt["date"].sort_values().index
    n_tr       = int(len(sorted_idx) * 0.80)
    tr_idx     = sorted_idx[:n_tr]
    te_idx     = sorted_idx[n_tr:]
    print(f"  Train: {len(tr_idx):,}   Test: {len(te_idx):,}   "
          f"(cutoff ≈ {df_fmt.loc[te_idx, 'date'].min().date()})")

    X_tr = X_all.loc[tr_idx]
    X_te = X_all.loc[te_idx]

    scaler  = StandardScaler()
    X_tr_s  = scaler.fit_transform(X_tr)
    X_te_s  = scaler.transform(X_te)

    # ── Targets dict ──────────────────────────────────────────────────────────
    TARGETS = {
        "ECO_DEV_form"   : (eco_dev_form,   eco_baseline_form,  "last5_same_fmt"),
        "ECO_DEV_career" : (eco_dev_career, career_eco,         "career_avg"),
        "BPI_DEV_form"   : (bpi_dev_form,   bpi_baseline_form,  "last5_BPI_same_fmt"),
        "BPI_DEV_career" : (bpi_dev_career, bpi_baseline_career,"career_BPI_avg"),
    }

    rows = []

    for tag, (y_all, naive_all, naive_label) in TARGETS.items():
        y_tr    = y_all.loc[tr_idx]
        y_te    = y_all.loc[te_idx]
        naive   = naive_all.loc[te_idx]

        # Drop any NaN rows in target
        mask_tr = ~y_tr.isna()
        mask_te = ~y_te.isna()
        nan_tr  = (~mask_tr).sum()
        nan_te  = (~mask_te).sum()

        print(f"\n  ── {tag}  (naive={naive_label})  "
              f"NaN drop: train={nan_tr}  test={nan_te}")
        print(f"     Target stats — mean={y_te[mask_te].mean():.3f}  "
              f"std={y_te[mask_te].std():.3f}  "
              f"min={y_te[mask_te].min():.3f}  "
              f"max={y_te[mask_te].max():.3f}")

        # Ridge
        rdg = Ridge(alpha=1.0)
        rdg.fit(X_tr_s[mask_tr], y_tr[mask_tr])
        yp_r = rdg.predict(X_te_s[mask_te])
        m_r  = reg_metrics(y_te[mask_te], yp_r, naive[mask_te], naive_label)
        rows.append({"Format": fmt, "Target": tag, "Model": "Ridge", **m_r})
        print(f"    Ridge    R²={m_r['R2']:+.4f}  MAE={m_r['MAE']:.4f}  "
              f"Naive_MAE={m_r['Naive_MAE']:.4f}  Uplift={m_r['Uplift_pct']:+.1f}%")

        # LightGBM
        lgm = lgb.LGBMRegressor(**LGB_PARAMS)
        lgm.fit(
            X_tr.loc[mask_tr], y_tr[mask_tr],
            eval_set=[(X_te.loc[mask_te], y_te[mask_te])],
            callbacks=[lgb.early_stopping(40, verbose=False),
                       lgb.log_evaluation(-1)],
        )
        yp_l = lgm.predict(X_te.loc[mask_te])
        m_l  = reg_metrics(y_te[mask_te], yp_l, naive[mask_te], naive_label)
        rows.append({"Format": fmt, "Target": tag, "Model": "LightGBM", **m_l})
        print(f"    LightGBM R²={m_l['R2']:+.4f}  MAE={m_l['MAE']:.4f}  "
              f"Naive_MAE={m_l['Naive_MAE']:.4f}  Uplift={m_l['Uplift_pct']:+.1f}%")

        save_shap(lgm, X_te.loc[mask_te], feat_cols, fmt, tag, OUTPUT_DIR)

    return pd.DataFrame(rows)


# ══════════════════════════════════════════════════════════════════════════════
# Step 7 — Summary + interpretation
# ══════════════════════════════════════════════════════════════════════════════

def print_summary(df):
    bar = "═" * 76
    print(f"\n\n{bar}")
    print("  FULL RESULTS TABLE")
    print(bar)
    print(df.to_string(index=False))

    # Best model per Format × Target
    best = df.loc[df.groupby(["Format", "Target"])["R2"].idxmax()]
    print(f"\n\n{bar}")
    print("  BEST MODEL PER FORMAT × TARGET")
    print(bar)
    print(best[["Format","Target","Model","R2","MAE","Naive_MAE",
                "Uplift_pct","Naive_src"]].to_string(index=False))

    # ── Cross-format R² comparison ──────────────────────────────────────────
    pivot = best.pivot_table(index="Target", columns="Format", values="R2")
    print(f"\n\n{bar}")
    print("  R² COMPARISON ACROSS FORMATS (best model per cell)")
    print(bar)
    print(pivot.to_string())

    # ── Uplift comparison ───────────────────────────────────────────────────
    pivot_u = best.pivot_table(index="Target", columns="Format", values="Uplift_pct")
    print(f"\n\n{bar}")
    print("  UPLIFT % OVER NAIVE BASELINE (best model per cell)")
    print(bar)
    print(pivot_u.to_string())

    print(f"""
{bar}
  INTERPRETATION GUIDE
{bar}

  DEVIATION TARGETS — what the numbers mean:
  ┌──────────────────────────────────────────────────────────────────────┐
  │ ECO_DEV_form    = eco - last5_same_format_eco                        │
  │   Positive → bowler went above (worse) than recent form              │
  │   Negative → bowler went below (better) than recent form             │
  │                                                                      │
  │ ECO_DEV_career  = eco - career_avg_eco                               │
  │   Measures single-match deviation from entire career baseline        │
  │                                                                      │
  │ BPI_DEV_form    = BPI - last5_same_format_BPI                        │
  │   Positive → better BPI than recent form                             │
  │                                                                      │
  │ BPI_DEV_career  = BPI - expanding_career_BPI_avg (shift-1)           │
  └──────────────────────────────────────────────────────────────────────┘

  WHAT A GOOD RESULT LOOKS LIKE:
  ┌──────────────────────────────────────────────────────────────────────┐
  │ R² > 0.20         Solid for sports; cite domain literature           │
  │ R² 0.10–0.20      Publishable with proper framing + baselines        │
  │ Uplift > 10%      Model meaningfully beats naive career-stat guessing│
  │ Uplift > 0%       At minimum, model is better than doing nothing     │
  │ Uplift < 0%       STOP — naive baseline is better; revisit features  │
  └──────────────────────────────────────────────────────────────────────┘

  PAPER FRAMING DECISION TABLE:
  ┌────────────────────────┬────────────────────────────────────────────┐
  │ Best result is         │ Suggested paper angle                      │
  ├────────────────────────┼────────────────────────────────────────────┤
  │ ECO_DEV_form > career  │ "In-format form is more predictive than    │
  │                        │  career averages" — contribution in itself  │
  ├────────────────────────┼────────────────────────────────────────────┤
  │ BPI_DEV better than    │ "Composite performance index is more        │
  │ ECO_DEV                │  predictable than single-outcome economy"   │
  ├────────────────────────┼────────────────────────────────────────────┤
  │ Test R² >> T20 R²      │ "Format affects predictability — Test       │
  │                        │  bowling is more consistent than T20"       │
  ├────────────────────────┼────────────────────────────────────────────┤
  │ SHAP: career_* dominate│ "Career history drives prediction more than │
  │                        │  match context or opponent quality"         │
  └────────────────────────┴────────────────────────────────────────────┘

  SHAP plots: {OUTPUT_DIR}
""")


# ══════════════════════════════════════════════════════════════════════════════
# Main
# ══════════════════════════════════════════════════════════════════════════════

def main():
    t0 = time.time()
    print("BowlerIQ Modelling Pipeline v2 — EcoDev + BPIDev")
    print(f"Data: {DATA_PATH}\n")

    df = pd.read_parquet(DATA_PATH)
    print(f"Loaded {df.shape[0]:,} rows × {df.shape[1]} columns")

    # Normalise match_type
    df["match_type"] = df["match_type"].str.strip().str.upper()
    df.loc[df["match_type"] == "T20I", "match_type"] = "T20"

    # Ensure date is datetime
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])
    print(f"match_type distribution:\n{df['match_type'].value_counts().to_string()}\n")

    # ── Engineer BPI rolling columns on FULL dataset (before format split) ──
    df = engineer_bpi_rolling(df)

    all_results = []

    for fmt in FORMATS:
        df_fmt = df[df["match_type"] == fmt.upper()].copy()
        if len(df_fmt) < 300:
            print(f"\nSkipping {fmt} — only {len(df_fmt)} rows")
            continue
        result_df = run_format(fmt, df_fmt)
        result_df.insert(0, "Format", fmt)
        # avoid duplicate Format col if already added inside run_format
        result_df = result_df.loc[:, ~result_df.columns.duplicated()]
        all_results.append(result_df)

    if not all_results:
        print("No formats processed. Check match_type values:")
        print(df["match_type"].value_counts())
        return

    full = pd.concat(all_results, ignore_index=True)
    out_csv = os.path.join(OUTPUT_DIR, "results_ecodev_bpidev.csv")
    full.to_csv(out_csv, index=False)
    print(f"\nCSV saved → {out_csv}")

    print_summary(full)
    print(f"Total runtime: {(time.time()-t0)/60:.1f} min")


if __name__ == "__main__":
    main()

BowlerIQ Modelling Pipeline v2 — EcoDev + BPIDev
Data: C:\Users\HPP\Bowleriq\abt_with_bpi_rowwise.parquet

Loaded 49,945 rows × 170 columns
match_type distribution:
match_type
T20     22917
ODI     19253
TEST     7775

  Engineering BPI rolling columns...
    _bpi_last5_form  NaN remaining: 3070
    _bpi_career_avg  NaN remaining: 3070

  FORMAT: T20   (22,917 rows)
  Eco baseline (form):   economy_last_5_t20  |  Career fallback used: 1681 rows
  BPI last-5 debut rows (→ career fallback used): 1701
  Features used: 61
  Train: 18,333   Test: 4,584   (cutoff ≈ 2024-11-26)

  ── ECO_DEV_form  (naive=last5_same_fmt)  NaN drop: train=0  test=0
     Target stats — mean=-0.646  std=3.093  min=-30.330  max=13.510
    Ridge    R²=+0.2481  MAE=2.0560  Naive_MAE=8.0539  Uplift=+74.5%
    LightGBM R²=+0.2567  MAE=2.0362  Naive_MAE=8.0539  Uplift=+74.7%
      SHAP → C:\Users\HPP\Bowleriq\modelling_results_v2\shap_T20_ECO_DEV_form.png

  ── ECO_DEV_career  (naive=career_avg)  NaN drop: train=0  tes

ValueError: cannot insert Format, already exists

In [13]:
"""
BowlerIQ — Modelling Pipeline v3 (5 Deviation Families)
=========================================================
10 target variants across 3 formats × 2 models = 60 model runs
Estimated runtime: ~12–18 minutes

Targets (all are: actual_value - baseline):
  ECO_DEV_form    = economy_rate       - last-5 format economy
  ECO_DEV_career  = economy_rate       - career_economy
  BPI_DEV_form    = BPI_Final          - last-5 format BPI
  BPI_DEV_career  = BPI_Final          - expanding career BPI avg
  WKT_DEV_form    = wickets            - last-5 format wickets
  WKT_DEV_career  = wickets            - career_wickets_per_match
  DOT_DEV_form    = dot_ball_pct       - last-5 format dot_ball_pct
  DOT_DEV_career  = dot_ball_pct       - career_dot_ball_pct
  STR_DEV_form    = strike_rate        - last-5 format strike_rate
  STR_DEV_career  = strike_rate        - career_strike_rate

Key design decisions:
  - Format-specific last-5 uses existing columns where available:
      economy  -> economy_last_5_t20/odi/test
      wickets  -> wickets_last_5_t20/odi/test
      strike_r -> strike_rate_last_5_t20/odi/test  (computed here if absent)
      dot_pct  -> dot_pct_last_5_t20/odi/test      (computed here if absent)
      BPI      -> computed rolling here (not in raw data)
  - Fallback chain for any NaN in last-5: career avg -> format median
  - All rolling computations use shift(1) to prevent leakage
  - Temporal 80/20 split on date (no random leakage)
  - Models: Ridge (linear baseline) + LightGBM (nonlinear)
  - SHAP: top-20 features saved per format x target
  - BUG FIX: removed duplicate result_df.insert("Format") in main()
"""

import warnings, os, time
warnings.filterwarnings("ignore")

import numpy  as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import shap

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute         import SimpleImputer
from sklearn.linear_model  import Ridge
from sklearn.metrics        import r2_score, mean_absolute_error, mean_squared_error
import lightgbm as lgb

# ── Config ────────────────────────────────────────────────────────────────────
DATA_PATH  = r"C:\Users\HPP\Bowleriq\abt_with_bpi_rowwise.parquet"
OUTPUT_DIR = r"C:\Users\HPP\Bowleriq\modelling_results_v3"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FORMATS    = ["T20", "ODI", "Test"]
RAND_STATE = 42

# ── Column taxonomy ───────────────────────────────────────────────────────────
LEAKAGE_OUTCOMES = [
    "balls_bowled","runs_conceded","wickets","dot_balls","total_extras",
    "wides","noballs","byes","legbyes","fours_conceded","sixes_conceded",
    "wickets_bowled","wickets_caught","wickets_lbw","wickets_stumped",
    "wickets_caught_and_bowled","wickets_hit_wicket",
    "balls_powerplay_t20","balls_middle_t20","balls_death_t20",
    "runs_powerplay_t20","runs_middle_t20","runs_death_t20",
    "wickets_powerplay_t20","wickets_middle_t20","wickets_death_t20",
    "dots_powerplay_t20","dots_death_t20",
    "balls_powerplay_odi","balls_middle_odi","balls_death_odi",
    "runs_powerplay_odi","runs_middle_odi","runs_death_odi",
    "wickets_powerplay_odi","wickets_middle_odi","wickets_death_odi",
    "balls_team_spell_1","balls_team_spell_2",
    "runs_team_spell_1","runs_team_spell_2",
    "wickets_team_spell_1","wickets_team_spell_2",
    "first_over","last_over","overs_bowled_distinct",
    "balls_to_rhb","balls_to_lhb",
    "economy_rate","strike_rate","bowling_average",
    "dot_ball_percentage","overs_bowled",
    "boundary_percentage","bowled_percentage","caught_percentage",
    "lbw_percentage","rhb_percentage","lhb_percentage",
    "dots_powerplay_pct_t20","dots_death_pct_t20",
    "winner","result_type","win_by_runs","win_by_wickets","win_method",
    "bowling_team_won",
]

LEAKAGE_BPI = [
    "economy_score","strike_score","wicket_score","dot_score",
    "BPI_Raw","Opposition_Factor","Opposition_Factor_Capped",
    "BPI_OppAdj_Capped","BPI_Final",
]

IDENTIFIERS = [
    "match_id","bowler","bowler_espn_id","bowler_full_name","bowler_dob",
    "venue_id","venue_original","venue","venue_canonical","venue_city",
    "bowling_team","opponent_team",
    "bowling_team_country","bowling_team_code","bowling_team_region",
    "opponent_team_country","opponent_team_code","opponent_team_region",
    "toss_winner","match_type","date",
    # target-construction helpers added by engineer_rolling()
    "economy_last_5",
    "economy_last_5_t20","economy_last_5_odi","economy_last_5_test",
    "_bpi_last5_form","_bpi_career_avg",
    "_wkt_last5_form","_wkt_career_avg",
    "_dot_last5_form","_dot_career_avg",
    "_str_last5_form","_str_career_avg",
]

ALL_DROP = set(LEAKAGE_OUTCOMES) | set(LEAKAGE_BPI) | set(IDENTIFIERS)

CAT_COLS = [
    "bowler_batting_style","bowler_bowling_style","toss_decision",
    "career_stage","bowling_team_type","bowling_team_member_type",
    "opponent_team_type","opponent_team_member_type",
    "bowler_metadata_quality","venue_country",
]

BOOL_COLS = [
    "is_home_match","bowling_team_won_toss","is_debut","economy_improving",
    "venue_specialist","opponent_specialist","first_time_at_venue",
    "first_time_vs_opponent","opponent_is_full_member","opponent_is_top8",
]

FORMAT_EXCLUSIVE = {
    "T20" : ["career_matches_t20","career_wickets_t20","career_economy_t20",
             "wickets_last_5_t20","economy_last_5_t20"],
    "ODI" : ["career_matches_odi","career_wickets_odi","career_economy_odi",
             "wickets_last_5_odi","economy_last_5_odi"],
    "Test": ["career_matches_test","career_wickets_test","career_economy_test",
             "wickets_last_5_test","economy_last_5_test"],
}

FORMAT_CROSS_DROP = {
    "T20" : set(FORMAT_EXCLUSIVE["ODI"])  | set(FORMAT_EXCLUSIVE["Test"]),
    "ODI" : set(FORMAT_EXCLUSIVE["T20"])  | set(FORMAT_EXCLUSIVE["Test"]),
    "Test": set(FORMAT_EXCLUSIVE["T20"])  | set(FORMAT_EXCLUSIVE["ODI"]),
}

LGB_PARAMS = dict(
    n_estimators=400, learning_rate=0.04, num_leaves=31,
    min_child_samples=30, subsample=0.8, colsample_bytree=0.8,
    random_state=RAND_STATE, n_jobs=-1, verbosity=-1,
)


# ══════════════════════════════════════════════════════════════════════════════
# Step 1 — Engineer all rolling baseline columns on full dataset
# ══════════════════════════════════════════════════════════════════════════════
def _rolling_last5_career(df, src_col):
    """
    Compute two shift-1 rolling features for src_col:
      last5_form : rolling(5, min_periods=1) per bowler x match_type, shift-1
      career_avg : expanding mean per bowler (cross-format), shift-1
    Returns (last5_series, career_series).
    NaN only for absolute debut rows (no prior history at all).
    """
    career = (
        df.groupby("bowler")[src_col]
          .transform(lambda s: s.shift(1).expanding().mean())
    )
    last5 = (
        df.groupby(["bowler", "match_type"])[src_col]
          .transform(lambda s: s.shift(1).rolling(window=5, min_periods=1).mean())
    )
    # Fallback: format debut row -> use cross-format career avg
    last5 = last5.fillna(career)
    return last5, career


def engineer_rolling(df):
    """
    Compute all rolling helper columns.
    Must be called ONCE on the full dataset before format split,
    so rolling history spans all formats correctly for career_avg.
    """
    print("  Engineering rolling baseline columns...")
    df = df.sort_values(["bowler", "match_type", "date"]).reset_index(drop=True)

    df["_bpi_last5_form"], df["_bpi_career_avg"] = _rolling_last5_career(df, "BPI_Final")
    df["_wkt_last5_form"], df["_wkt_career_avg"] = _rolling_last5_career(df, "wickets")
    df["_dot_last5_form"], df["_dot_career_avg"] = _rolling_last5_career(df, "dot_ball_percentage")
    df["_str_last5_form"], df["_str_career_avg"] = _rolling_last5_career(df, "strike_rate")

    for col in ["_bpi_last5_form","_bpi_career_avg",
                "_wkt_last5_form","_wkt_career_avg",
                "_dot_last5_form","_dot_career_avg",
                "_str_last5_form","_str_career_avg"]:
        n = df[col].isna().sum()
        if n > 0:
            print(f"    {col:25s}  NaN remaining (absolute debuts): {n}")

    return df


# ══════════════════════════════════════════════════════════════════════════════
# Step 2 — Encoding helpers
# ══════════════════════════════════════════════════════════════════════════════
def encode_and_cast(df):
    for col in CAT_COLS:
        if col in df.columns:
            df[col] = df[col].fillna("__MISSING__").astype(str)
            df[col] = LabelEncoder().fit_transform(df[col])
    for col in BOOL_COLS:
        if col in df.columns:
            df[col] = (
                df[col].fillna(0)
                       .map(lambda x: 1 if str(x).strip().lower() in ("1","true","yes") else 0)
                       .astype(int)
            )
    return df


# ══════════════════════════════════════════════════════════════════════════════
# Step 3 — Feature matrix builder
# ══════════════════════════════════════════════════════════════════════════════
def build_features(df, fmt):
    drop_here = ALL_DROP | FORMAT_CROSS_DROP[fmt]
    feat_cols = [
        c for c in df.columns
        if c not in drop_here and df[c].dtype != object
    ]
    X = df[feat_cols].copy()
    imp = SimpleImputer(strategy="median")
    X_arr = imp.fit_transform(X)
    return pd.DataFrame(X_arr, columns=feat_cols, index=df.index), feat_cols


# ══════════════════════════════════════════════════════════════════════════════
# Step 4 — Metrics
# ══════════════════════════════════════════════════════════════════════════════
def reg_metrics(y_true, y_pred, y_naive, naive_label):
    r2      = r2_score(y_true, y_pred)
    mae     = mean_absolute_error(y_true, y_pred)
    rmse    = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    naive_m = mean_absolute_error(y_true, y_naive)
    uplift  = (naive_m - mae) / naive_m * 100 if naive_m > 0 else 0.0
    return dict(
        R2        = round(r2,     4),
        MAE       = round(mae,    4),
        RMSE      = round(rmse,   4),
        Naive_MAE = round(naive_m, 4),
        Naive_src = naive_label,
        Uplift_pct= round(uplift,  2),
    )


# ══════════════════════════════════════════════════════════════════════════════
# Step 5 — SHAP plot
# ══════════════════════════════════════════════════════════════════════════════
def save_shap(model, X_te, feat_cols, fmt, tag, out_dir):
    try:
        X_s  = X_te.sample(min(600, len(X_te)), random_state=RAND_STATE)
        exp  = shap.TreeExplainer(model)
        sv   = exp.shap_values(X_s)
        if not isinstance(sv, np.ndarray):
            sv = np.array(sv)
        if sv.ndim == 3:
            sv = np.mean(np.abs(sv), axis=0)
        mean_abs = np.abs(sv).mean(axis=0)
        n     = min(len(feat_cols), len(mean_abs))
        cols_ = feat_cols[:n]
        vals_ = mean_abs[:n]
        idx   = np.argsort(vals_)[-20:][::-1]
        fig, ax = plt.subplots(figsize=(9, 6))
        ax.barh([cols_[i] for i in idx[::-1]],
                [vals_[i] for i in idx[::-1]],
                color="steelblue")
        ax.set_xlabel("Mean |SHAP value|")
        ax.set_title(f"Top-20 Features — {fmt} / {tag}")
        plt.tight_layout()
        fpath = os.path.join(out_dir, f"shap_{fmt}_{tag}.png")
        plt.savefig(fpath, dpi=120)
        plt.close()
        print(f"      SHAP -> {fpath}")
    except Exception as e:
        print(f"      SHAP skipped ({e})")


# ══════════════════════════════════════════════════════════════════════════════
# Step 6 — Baseline builder with fallback chain
# ══════════════════════════════════════════════════════════════════════════════
def make_baseline(primary, fallback_career, global_median):
    """
    Fallback chain:
      primary (last-5 format col or rolling col)
        -> fallback_career (career average)
          -> global_median (format-level median, for absolute debuts)
    """
    b = primary.copy()
    b = b.fillna(fallback_career)
    b = b.fillna(global_median)
    return b


# ══════════════════════════════════════════════════════════════════════════════
# Step 7 — Per-format pipeline
# ══════════════════════════════════════════════════════════════════════════════
def run_format(fmt, df_fmt):
    print(f"\n{'='*65}")
    print(f"  FORMAT: {fmt}   ({len(df_fmt):,} rows)")
    print(f"{'='*65}")

    fmt_lower = fmt.lower()
    df_fmt = encode_and_cast(df_fmt.copy())

    # ── Raw outcome columns ───────────────────────────────────────────────────
    eco_raw = df_fmt["economy_rate"].copy()
    bpi_raw = df_fmt["BPI_Final"].copy()
    wkt_raw = df_fmt["wickets"].copy()
    dot_raw = df_fmt["dot_ball_percentage"].copy()
    str_raw = df_fmt["strike_rate"].copy()

    # ── Career average columns ────────────────────────────────────────────────
    career_eco = df_fmt["career_economy"].copy()
    career_wkt = df_fmt["career_wickets_per_match"].copy()
    career_dot = df_fmt["career_dot_ball_pct"].copy()
    career_str = df_fmt["career_strike_rate"].copy()

    # ── Format-specific last-5 column names ───────────────────────────────────
    eco_last5_col = f"economy_last_5_{fmt_lower}"
    wkt_last5_col = f"wickets_last_5_{fmt_lower}"
    str_last5_col = f"strike_rate_last_5_{fmt_lower}"
    dot_last5_col = f"dot_pct_last_5_{fmt_lower}"

    def _get_col(col_name, career_fallback):
        if col_name in df_fmt.columns:
            return df_fmt[col_name].copy()
        print(f"    INFO: {col_name} not in data -> using career avg as form baseline")
        return career_fallback.copy()

    # ── Format-level medians for absolute-debut fallback ──────────────────────
    med_eco = career_eco.median()
    med_wkt = career_wkt.median()
    med_dot = career_dot.median()
    med_str = career_str.median()
    med_bpi = df_fmt["_bpi_career_avg"].median()

    # ── Build all 10 deviation targets ───────────────────────────────────────
    # ECO_DEV
    eco_bl_form   = make_baseline(_get_col(eco_last5_col, career_eco), career_eco, med_eco)
    eco_bl_career = career_eco.fillna(med_eco)
    eco_dev_form   = eco_raw - eco_bl_form
    eco_dev_career = eco_raw - eco_bl_career

    # BPI_DEV
    bpi_bl_form   = make_baseline(df_fmt["_bpi_last5_form"], df_fmt["_bpi_career_avg"], med_bpi)
    bpi_bl_career = df_fmt["_bpi_career_avg"].fillna(med_bpi)
    bpi_dev_form   = bpi_raw - bpi_bl_form
    bpi_dev_career = bpi_raw - bpi_bl_career

    # WKT_DEV
    wkt_bl_form   = make_baseline(_get_col(wkt_last5_col, career_wkt),
                                   df_fmt["_wkt_career_avg"], med_wkt)
    wkt_bl_career = df_fmt["_wkt_career_avg"].fillna(med_wkt)
    wkt_dev_form   = wkt_raw - wkt_bl_form
    wkt_dev_career = wkt_raw - wkt_bl_career

    # DOT_DEV
    dot_bl_form   = make_baseline(_get_col(dot_last5_col, career_dot),
                                   df_fmt["_dot_career_avg"], med_dot)
    dot_bl_career = df_fmt["_dot_career_avg"].fillna(med_dot)
    dot_dev_form   = dot_raw - dot_bl_form
    dot_dev_career = dot_raw - dot_bl_career

    # STR_DEV
    str_bl_form   = make_baseline(_get_col(str_last5_col, career_str),
                                   df_fmt["_str_career_avg"], med_str)
    str_bl_career = df_fmt["_str_career_avg"].fillna(med_str)
    str_dev_form   = str_raw - str_bl_form
    str_dev_career = str_raw - str_bl_career

    # ── Info ──────────────────────────────────────────────────────────────────
    for col_name in [eco_last5_col, wkt_last5_col, str_last5_col, dot_last5_col]:
        status = "FOUND" if col_name in df_fmt.columns else "MISSING -> career fallback"
        print(f"  {col_name:35s}: {status}")

    # ── Feature matrix ────────────────────────────────────────────────────────
    X_all, feat_cols = build_features(df_fmt, fmt)
    print(f"  Features used: {len(feat_cols)}")

    # ── Temporal split (80/20) ────────────────────────────────────────────────
    sorted_idx = df_fmt["date"].sort_values().index
    n_tr   = int(len(sorted_idx) * 0.80)
    tr_idx = sorted_idx[:n_tr]
    te_idx = sorted_idx[n_tr:]
    print(f"  Train: {len(tr_idx):,}   Test: {len(te_idx):,}   "
          f"(cutoff ~ {df_fmt.loc[te_idx, 'date'].min().date()})")

    X_tr = X_all.loc[tr_idx]
    X_te = X_all.loc[te_idx]
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)

    # ── Target registry ───────────────────────────────────────────────────────
    TARGETS = {
        "ECO_DEV_form"   : (eco_dev_form,   eco_bl_form,    "last5_eco_fmt"),
        "ECO_DEV_career" : (eco_dev_career, eco_bl_career,  "career_eco"),
        "BPI_DEV_form"   : (bpi_dev_form,   bpi_bl_form,    "last5_BPI_fmt"),
        "BPI_DEV_career" : (bpi_dev_career, bpi_bl_career,  "career_BPI"),
        "WKT_DEV_form"   : (wkt_dev_form,   wkt_bl_form,    "last5_wkt_fmt"),
        "WKT_DEV_career" : (wkt_dev_career, wkt_bl_career,  "career_wkt_per_match"),
        "DOT_DEV_form"   : (dot_dev_form,   dot_bl_form,    "last5_dot_pct_fmt"),
        "DOT_DEV_career" : (dot_dev_career, dot_bl_career,  "career_dot_pct"),
        "STR_DEV_form"   : (str_dev_form,   str_bl_form,    "last5_str_fmt"),
        "STR_DEV_career" : (str_dev_career, str_bl_career,  "career_str"),
    }

    rows = []
    for tag, (y_all, naive_all, naive_label) in TARGETS.items():
        y_tr  = y_all.loc[tr_idx]
        y_te  = y_all.loc[te_idx]
        naive = naive_all.loc[te_idx]

        mask_tr = ~y_tr.isna()
        mask_te = ~y_te.isna()
        nan_tr  = (~mask_tr).sum()
        nan_te  = (~mask_te).sum()

        print(f"\n  -- {tag}  (naive={naive_label})  "
              f"NaN drop: train={nan_tr}  test={nan_te}")
        print(f"     Target -- mean={y_te[mask_te].mean():.3f}  "
              f"std={y_te[mask_te].std():.3f}  "
              f"min={y_te[mask_te].min():.3f}  "
              f"max={y_te[mask_te].max():.3f}")

        # Ridge
        rdg = Ridge(alpha=1.0)
        rdg.fit(X_tr_s[mask_tr], y_tr[mask_tr])
        yp_r = rdg.predict(X_te_s[mask_te])
        m_r  = reg_metrics(y_te[mask_te], yp_r, naive[mask_te], naive_label)
        rows.append({"Format": fmt, "Target": tag, "Model": "Ridge", **m_r})
        print(f"    Ridge    R2={m_r['R2']:+.4f}  MAE={m_r['MAE']:.4f}  "
              f"Naive_MAE={m_r['Naive_MAE']:.4f}  Uplift={m_r['Uplift_pct']:+.1f}%")

        # LightGBM
        lgm = lgb.LGBMRegressor(**LGB_PARAMS)
        lgm.fit(
            X_tr.loc[mask_tr], y_tr[mask_tr],
            eval_set=[(X_te.loc[mask_te], y_te[mask_te])],
            callbacks=[lgb.early_stopping(40, verbose=False),
                       lgb.log_evaluation(-1)],
        )
        yp_l = lgm.predict(X_te.loc[mask_te])
        m_l  = reg_metrics(y_te[mask_te], yp_l, naive[mask_te], naive_label)
        rows.append({"Format": fmt, "Target": tag, "Model": "LightGBM", **m_l})
        print(f"    LightGBM R2={m_l['R2']:+.4f}  MAE={m_l['MAE']:.4f}  "
              f"Naive_MAE={m_l['Naive_MAE']:.4f}  Uplift={m_l['Uplift_pct']:+.1f}%")

        save_shap(lgm, X_te.loc[mask_te], feat_cols, fmt, tag, OUTPUT_DIR)

    # Format col is already inside each row dict -- no insert() needed
    return pd.DataFrame(rows)


# ══════════════════════════════════════════════════════════════════════════════
# Step 8 — Summary + interpretation
# ══════════════════════════════════════════════════════════════════════════════
def print_summary(df):
    bar = "=" * 80

    print(f"\n\n{bar}")
    print("  FULL RESULTS TABLE")
    print(bar)
    print(df.to_string(index=False))

    best = df.loc[df.groupby(["Format","Target"])["R2"].idxmax()]

    print(f"\n\n{bar}")
    print("  BEST MODEL PER FORMAT x TARGET")
    print(bar)
    print(best[["Format","Target","Model","R2","MAE","Naive_MAE",
                "Uplift_pct","Naive_src"]].to_string(index=False))

    pivot_r2 = best.pivot_table(index="Target", columns="Format", values="R2")
    print(f"\n\n{bar}")
    print("  R2 ACROSS FORMATS (best model per cell)")
    print(bar)
    print(pivot_r2.to_string())

    pivot_up = best.pivot_table(index="Target", columns="Format", values="Uplift_pct")
    print(f"\n\n{bar}")
    print("  UPLIFT % OVER NAIVE BASELINE (best model per cell)")
    print(bar)
    print(pivot_up.to_string())

    print(f"""
{bar}
  INTERPRETATION GUIDE
{bar}
  DEVIATION TARGETS EXPLAINED:
    ECO_DEV_form    economy_rate - last5_format_eco
      Positive -> conceded more than recent form; Negative -> more economical
    BPI_DEV_form    BPI_Final - last5_format_BPI
      Positive -> outperformed recent composite form
    WKT_DEV_form    wickets - last5_format_wickets
      Positive -> took more wickets than recent form predicts
    DOT_DEV_form    dot_ball_pct - last5_format_dot_pct
      Positive -> bowled more dot balls than recent form
    STR_DEV_form    strike_rate - last5_format_strike_rate
      Positive -> took wickets less frequently than recent form
    *_career variants compare to expanding career avg (not just last-5)

  WHAT A GOOD RESULT LOOKS LIKE:
    R2 > 0.20         Solid for sports analytics
    R2 0.10-0.20      Publishable with proper framing + baselines
    Uplift > 10%      Model meaningfully beats naive career-stat guessing
    Uplift > 0%       Model is at least better than doing nothing
    Uplift < 0%       STOP -- naive baseline is better; revisit features

  PAPER FRAMING DECISION TABLE:
    ECO_DEV > BPI_DEV        -> Economy deviation is most predictable
    BPI_DEV > ECO_DEV        -> Composite index is more predictable
    WKT_DEV R2 near 0        -> Wicket-taking is near-random (expected)
    form > career (all tgts) -> Recent form beats career history
    Test R2 >> T20 R2        -> Format consistency shapes predictability
    SHAP: career_* dominant  -> Career history > match context

  SHAP plots saved to: {OUTPUT_DIR}
""")


# ══════════════════════════════════════════════════════════════════════════════
# Main
# ══════════════════════════════════════════════════════════════════════════════
def main():
    t0 = time.time()
    print("BowlerIQ Modelling Pipeline v3 -- 5 Deviation Families")
    print(f"Data: {DATA_PATH}\n")

    df = pd.read_parquet(DATA_PATH)
    print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")

    df["match_type"] = df["match_type"].str.strip().str.upper()
    df.loc[df["match_type"] == "T20I", "match_type"] = "T20"

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])

    print(f"match_type distribution:\n{df['match_type'].value_counts().to_string()}\n")

    # Must run on full dataset before format split
    df = engineer_rolling(df)

    all_results = []
    for fmt in FORMATS:
        df_fmt = df[df["match_type"] == fmt.upper()].copy()
        if len(df_fmt) < 300:
            print(f"\nSkipping {fmt} -- only {len(df_fmt)} rows")
            continue

        result_df = run_format(fmt, df_fmt)

        # BUG FIX: run_format already puts "Format" in each row dict.
        # The old code did result_df.insert(0, "Format", fmt) here, which
        # caused ValueError: cannot insert Format, already exists.
        # Simply removed -- no insert needed.
        result_df = result_df.loc[:, ~result_df.columns.duplicated()]
        all_results.append(result_df)

    if not all_results:
        print("No formats processed. Check match_type values:")
        print(df["match_type"].value_counts())
        return

    full = pd.concat(all_results, ignore_index=True)
    out_csv = os.path.join(OUTPUT_DIR, "results_all_devs_v3.csv")
    full.to_csv(out_csv, index=False)
    print(f"\nCSV saved -> {out_csv}")

    print_summary(full)
    print(f"Total runtime: {(time.time()-t0)/60:.1f} min")


if __name__ == "__main__":
    main()

BowlerIQ Modelling Pipeline v3 -- 5 Deviation Families
Data: C:\Users\HPP\Bowleriq\abt_with_bpi_rowwise.parquet

Loaded 49,945 rows x 170 columns
match_type distribution:
match_type
T20     22917
ODI     19253
TEST     7775

  Engineering rolling baseline columns...
    _bpi_last5_form            NaN remaining (absolute debuts): 3070
    _bpi_career_avg            NaN remaining (absolute debuts): 3070
    _wkt_last5_form            NaN remaining (absolute debuts): 3070
    _wkt_career_avg            NaN remaining (absolute debuts): 3070
    _dot_last5_form            NaN remaining (absolute debuts): 3070
    _dot_career_avg            NaN remaining (absolute debuts): 3070
    _str_last5_form            NaN remaining (absolute debuts): 3070
    _str_career_avg            NaN remaining (absolute debuts): 3070

  FORMAT: T20   (22,917 rows)
    INFO: dot_pct_last_5_t20 not in data -> using career avg as form baseline
    INFO: strike_rate_last_5_t20 not in data -> using career avg as form

In [1]:
"""
BowlerIQ — Modelling Pipeline v3 + WKT Fix
============================================
Identical to v3 with one targeted change:

  FIX — WKT_DEV target feature contamination
  ─────────────────────────────────────────────
  Problem:  wickets_last_5_{fmt} is kept as a feature (via FORMAT_EXCLUSIVE)
            AND is the exact column used as the WKT_DEV_form baseline.
            The model trivially learns:  prediction ≈ -wickets_last_5 + C
            inflating R² to ~0.92 (artifact, not genuine predictive signal).

  Solution: When building the feature matrix for ANY WKT_DEV target,
            additionally drop wickets_last_5_{fmt} from the feature set.
            This is implemented via the extra_drop argument to build_features().

  Unaffected targets: ECO_DEV, BPI_DEV, DOT_DEV, STR_DEV — unchanged.

Output directory: modelling_results_v3_wktfix
"""

import warnings, os, time
warnings.filterwarnings("ignore")

import numpy  as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import shap

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute         import SimpleImputer
from sklearn.linear_model  import Ridge
from sklearn.metrics        import r2_score, mean_absolute_error, mean_squared_error
import lightgbm as lgb

# ── Config ────────────────────────────────────────────────────────────────────
DATA_PATH  = r"C:\Users\HPP\Bowleriq\abt_with_bpi_rowwise.parquet"
OUTPUT_DIR = r"C:\Users\HPP\Bowleriq\modelling_results_v3_wktfix"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FORMATS    = ["T20", "ODI", "Test"]
RAND_STATE = 42

# ── Column taxonomy ───────────────────────────────────────────────────────────
LEAKAGE_OUTCOMES = [
    "balls_bowled","runs_conceded","wickets","dot_balls","total_extras",
    "wides","noballs","byes","legbyes","fours_conceded","sixes_conceded",
    "wickets_bowled","wickets_caught","wickets_lbw","wickets_stumped",
    "wickets_caught_and_bowled","wickets_hit_wicket",
    "balls_powerplay_t20","balls_middle_t20","balls_death_t20",
    "runs_powerplay_t20","runs_middle_t20","runs_death_t20",
    "wickets_powerplay_t20","wickets_middle_t20","wickets_death_t20",
    "dots_powerplay_t20","dots_death_t20",
    "balls_powerplay_odi","balls_middle_odi","balls_death_odi",
    "runs_powerplay_odi","runs_middle_odi","runs_death_odi",
    "wickets_powerplay_odi","wickets_middle_odi","wickets_death_odi",
    "balls_team_spell_1","balls_team_spell_2",
    "runs_team_spell_1","runs_team_spell_2",
    "wickets_team_spell_1","wickets_team_spell_2",
    "first_over","last_over","overs_bowled_distinct",
    "balls_to_rhb","balls_to_lhb",
    "economy_rate","strike_rate","bowling_average",
    "dot_ball_percentage","overs_bowled",
    "boundary_percentage","bowled_percentage","caught_percentage",
    "lbw_percentage","rhb_percentage","lhb_percentage",
    "dots_powerplay_pct_t20","dots_death_pct_t20",
    "winner","result_type","win_by_runs","win_by_wickets","win_method",
    "bowling_team_won",
]

LEAKAGE_BPI = [
    "economy_score","strike_score","wicket_score","dot_score",
    "BPI_Raw","Opposition_Factor","Opposition_Factor_Capped",
    "BPI_OppAdj_Capped","BPI_Final",
]

IDENTIFIERS = [
    "match_id","bowler","bowler_espn_id","bowler_full_name","bowler_dob",
    "venue_id","venue_original","venue","venue_canonical","venue_city",
    "bowling_team","opponent_team",
    "bowling_team_country","bowling_team_code","bowling_team_region",
    "opponent_team_country","opponent_team_code","opponent_team_region",
    "toss_winner","match_type","date",
    "economy_last_5",
    "economy_last_5_t20","economy_last_5_odi","economy_last_5_test",
    "_bpi_last5_form","_bpi_career_avg",
    "_wkt_last5_form","_wkt_career_avg",
    "_dot_last5_form","_dot_career_avg",
    "_str_last5_form","_str_career_avg",
]

ALL_DROP = set(LEAKAGE_OUTCOMES) | set(LEAKAGE_BPI) | set(IDENTIFIERS)

CAT_COLS = [
    "bowler_batting_style","bowler_bowling_style","toss_decision",
    "career_stage","bowling_team_type","bowling_team_member_type",
    "opponent_team_type","opponent_team_member_type",
    "bowler_metadata_quality","venue_country",
]

BOOL_COLS = [
    "is_home_match","bowling_team_won_toss","is_debut","economy_improving",
    "venue_specialist","opponent_specialist","first_time_at_venue",
    "first_time_vs_opponent","opponent_is_full_member","opponent_is_top8",
]

FORMAT_EXCLUSIVE = {
    "T20" : ["career_matches_t20","career_wickets_t20","career_economy_t20",
             "wickets_last_5_t20","economy_last_5_t20"],
    "ODI" : ["career_matches_odi","career_wickets_odi","career_economy_odi",
             "wickets_last_5_odi","economy_last_5_odi"],
    "Test": ["career_matches_test","career_wickets_test","career_economy_test",
             "wickets_last_5_test","economy_last_5_test"],
}

FORMAT_CROSS_DROP = {
    "T20" : set(FORMAT_EXCLUSIVE["ODI"])  | set(FORMAT_EXCLUSIVE["Test"]),
    "ODI" : set(FORMAT_EXCLUSIVE["T20"])  | set(FORMAT_EXCLUSIVE["Test"]),
    "Test": set(FORMAT_EXCLUSIVE["T20"])  | set(FORMAT_EXCLUSIVE["ODI"]),
}

# ── WKT fix: columns to additionally drop for WKT_DEV targets ─────────────────
# These are the exact columns used as the WKT_DEV_form baseline.
# Including them as features creates a near-tautological prediction.
# Keyed by format string (upper) for convenience in run_format().
WKT_EXTRA_DROP = {
    "T20" : {"wickets_last_5_t20"},
    "ODI" : {"wickets_last_5_odi"},
    "Test": {"wickets_last_5_test"},
}

LGB_PARAMS = dict(
    n_estimators=400, learning_rate=0.04, num_leaves=31,
    min_child_samples=30, subsample=0.8, colsample_bytree=0.8,
    random_state=RAND_STATE, n_jobs=-1, verbosity=-1,
)


# ══════════════════════════════════════════════════════════════════════════════
# Rolling feature engineering (unchanged from v3)
# ══════════════════════════════════════════════════════════════════════════════
def _rolling_last5_career(df, src_col):
    career = (
        df.groupby("bowler")[src_col]
          .transform(lambda s: s.shift(1).expanding().mean())
    )
    last5 = (
        df.groupby(["bowler", "match_type"])[src_col]
          .transform(lambda s: s.shift(1).rolling(window=5, min_periods=1).mean())
    )
    return last5.fillna(career), career


def engineer_rolling(df):
    print("  Engineering rolling baseline columns...")
    df = df.sort_values(["bowler", "match_type", "date"]).reset_index(drop=True)
    df["_bpi_last5_form"], df["_bpi_career_avg"] = _rolling_last5_career(df, "BPI_Final")
    df["_wkt_last5_form"], df["_wkt_career_avg"] = _rolling_last5_career(df, "wickets")
    df["_dot_last5_form"], df["_dot_career_avg"] = _rolling_last5_career(df, "dot_ball_percentage")
    df["_str_last5_form"], df["_str_career_avg"] = _rolling_last5_career(df, "strike_rate")
    for col in ["_bpi_last5_form","_bpi_career_avg","_wkt_last5_form","_wkt_career_avg",
                "_dot_last5_form","_dot_career_avg","_str_last5_form","_str_career_avg"]:
        n = df[col].isna().sum()
        if n > 0:
            print(f"    {col:25s}  NaN remaining (absolute debuts): {n}")
    return df


# ══════════════════════════════════════════════════════════════════════════════
# Encoding
# ══════════════════════════════════════════════════════════════════════════════
def encode_and_cast(df):
    for col in CAT_COLS:
        if col in df.columns:
            df[col] = df[col].fillna("__MISSING__").astype(str)
            df[col] = LabelEncoder().fit_transform(df[col])
    for col in BOOL_COLS:
        if col in df.columns:
            df[col] = (df[col].fillna(0)
                              .map(lambda x: 1 if str(x).strip().lower() in ("1","true","yes") else 0)
                              .astype(int))
    return df


# ══════════════════════════════════════════════════════════════════════════════
# Feature matrix builder — now accepts extra_drop for per-target exclusions
# ══════════════════════════════════════════════════════════════════════════════
def build_features(df, fmt, extra_drop=frozenset()):
    """
    extra_drop: additional column names to exclude beyond the standard taxonomy.
    Used to remove WKT baseline columns when predicting WKT_DEV targets.
    """
    drop_here = ALL_DROP | FORMAT_CROSS_DROP[fmt] | set(extra_drop)
    feat_cols = [
        c for c in df.columns
        if c not in drop_here and df[c].dtype != object
    ]
    X = df[feat_cols].copy()
    imp = SimpleImputer(strategy="median")
    X_arr = imp.fit_transform(X)
    return pd.DataFrame(X_arr, columns=feat_cols, index=df.index), feat_cols


# ══════════════════════════════════════════════════════════════════════════════
# Metrics, SHAP, baseline builder (unchanged from v3)
# ══════════════════════════════════════════════════════════════════════════════
def reg_metrics(y_true, y_pred, y_naive, naive_label):
    r2      = r2_score(y_true, y_pred)
    mae     = mean_absolute_error(y_true, y_pred)
    rmse    = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    naive_m = mean_absolute_error(y_true, y_naive)
    uplift  = (naive_m - mae) / naive_m * 100 if naive_m > 0 else 0.0
    return dict(R2=round(r2,4), MAE=round(mae,4), RMSE=round(rmse,4),
                Naive_MAE=round(naive_m,4), Naive_src=naive_label,
                Uplift_pct=round(uplift,2))


def save_shap(model, X_te, feat_cols, fmt, tag, out_dir, prefix="lgb"):
    try:
        X_s = X_te.sample(min(600, len(X_te)), random_state=RAND_STATE)
        exp = shap.TreeExplainer(model)
        sv  = exp.shap_values(X_s)
        if not isinstance(sv, np.ndarray): sv = np.array(sv)
        if sv.ndim == 3: sv = np.mean(np.abs(sv), axis=0)
        mean_abs = np.abs(sv).mean(axis=0)
        n     = min(len(feat_cols), len(mean_abs))
        cols_ = feat_cols[:n]
        vals_ = mean_abs[:n]
        idx   = np.argsort(vals_)[-20:][::-1]
        fig, ax = plt.subplots(figsize=(9, 6))
        ax.barh([cols_[i] for i in idx[::-1]], [vals_[i] for i in idx[::-1]], color="steelblue")
        ax.set_xlabel("Mean |SHAP value|")
        ax.set_title(f"Top-20 Features — {fmt} / {tag}")
        plt.tight_layout()
        fpath = os.path.join(out_dir, f"shap_{prefix}_{fmt}_{tag}.png")
        plt.savefig(fpath, dpi=120); plt.close()
        print(f"      SHAP -> {fpath}")
    except Exception as e:
        print(f"      SHAP skipped ({e})")


def make_baseline(primary, fallback_career, global_median):
    return primary.copy().fillna(fallback_career).fillna(global_median)


# ══════════════════════════════════════════════════════════════════════════════
# Per-format pipeline
# ══════════════════════════════════════════════════════════════════════════════
def run_format(fmt, df_fmt):
    print(f"\n{'='*65}")
    print(f"  FORMAT: {fmt}   ({len(df_fmt):,} rows)")
    print(f"{'='*65}")

    fmt_lower  = fmt.lower()
    df_fmt     = encode_and_cast(df_fmt.copy())
    wkt_drop   = WKT_EXTRA_DROP.get(fmt, set())  # format-specific WKT correlated cols

    eco_raw  = df_fmt["economy_rate"].copy()
    bpi_raw  = df_fmt["BPI_Final"].copy()
    wkt_raw  = df_fmt["wickets"].copy()
    dot_raw  = df_fmt["dot_ball_percentage"].copy()
    str_raw  = df_fmt["strike_rate"].copy()

    career_eco = df_fmt["career_economy"].copy()
    career_wkt = df_fmt["career_wickets_per_match"].copy()
    career_dot = df_fmt["career_dot_ball_pct"].copy()
    career_str = df_fmt["career_strike_rate"].copy()

    eco_last5_col = f"economy_last_5_{fmt_lower}"
    wkt_last5_col = f"wickets_last_5_{fmt_lower}"
    str_last5_col = f"strike_rate_last_5_{fmt_lower}"
    dot_last5_col = f"dot_pct_last_5_{fmt_lower}"

    def _get_col(col_name, career_fallback):
        if col_name in df_fmt.columns:
            return df_fmt[col_name].copy()
        print(f"    INFO: {col_name} not in data -> career avg used as form baseline")
        return career_fallback.copy()

    med_eco = career_eco.median(); med_wkt = career_wkt.median()
    med_dot = career_dot.median(); med_str = career_str.median()
    med_bpi = df_fmt["_bpi_career_avg"].median()

    eco_bl_form   = make_baseline(_get_col(eco_last5_col, career_eco), career_eco, med_eco)
    eco_bl_career = career_eco.fillna(med_eco)
    bpi_bl_form   = make_baseline(df_fmt["_bpi_last5_form"], df_fmt["_bpi_career_avg"], med_bpi)
    bpi_bl_career = df_fmt["_bpi_career_avg"].fillna(med_bpi)
    wkt_bl_form   = make_baseline(_get_col(wkt_last5_col, career_wkt), df_fmt["_wkt_career_avg"], med_wkt)
    wkt_bl_career = df_fmt["_wkt_career_avg"].fillna(med_wkt)
    dot_bl_form   = make_baseline(_get_col(dot_last5_col, career_dot), df_fmt["_dot_career_avg"], med_dot)
    dot_bl_career = df_fmt["_dot_career_avg"].fillna(med_dot)
    str_bl_form   = make_baseline(_get_col(str_last5_col, career_str), df_fmt["_str_career_avg"], med_str)
    str_bl_career = df_fmt["_str_career_avg"].fillna(med_str)

    TARGETS = {
        "ECO_DEV_form"   : (eco_raw - eco_bl_form,   eco_bl_form,    "last5_eco_fmt",       False),
        "ECO_DEV_career" : (eco_raw - eco_bl_career,  eco_bl_career,  "career_eco",           False),
        "BPI_DEV_form"   : (bpi_raw - bpi_bl_form,   bpi_bl_form,    "last5_BPI_fmt",        False),
        "BPI_DEV_career" : (bpi_raw - bpi_bl_career,  bpi_bl_career,  "career_BPI",           False),
        "WKT_DEV_form"   : (wkt_raw - wkt_bl_form,   wkt_bl_form,    "last5_wkt_fmt",        True),
        "WKT_DEV_career" : (wkt_raw - wkt_bl_career,  wkt_bl_career,  "career_wkt_per_match", True),
        "DOT_DEV_form"   : (dot_raw - dot_bl_form,   dot_bl_form,    "last5_dot_pct_fmt",    False),
        "DOT_DEV_career" : (dot_raw - dot_bl_career,  dot_bl_career,  "career_dot_pct",       False),
        "STR_DEV_form"   : (str_raw - str_bl_form,   str_bl_form,    "last5_str_fmt",        False),
        "STR_DEV_career" : (str_raw - str_bl_career,  str_bl_career,  "career_str",           False),
    }
    # 4th element: is_wkt_target — drives whether wkt_drop is applied

    # Temporal split indices (same across all targets for consistency)
    sorted_idx = df_fmt["date"].sort_values().index
    n_tr   = int(len(sorted_idx) * 0.80)
    tr_idx = sorted_idx[:n_tr]
    te_idx = sorted_idx[n_tr:]
    print(f"  Train: {len(tr_idx):,}   Test: {len(te_idx):,}   "
          f"(cutoff ~ {df_fmt.loc[te_idx, 'date'].min().date()})")

    rows = []
    for tag, (y_all, naive_all, naive_label, is_wkt) in TARGETS.items():

        # ── Build feature matrix — drop wkt correlated cols for WKT targets ──
        extra = wkt_drop if is_wkt else frozenset()
        X_all, feat_cols = build_features(df_fmt, fmt, extra_drop=extra)

        if is_wkt:
            dropped = wkt_drop & set(df_fmt.columns)
            print(f"\n  -- {tag}  [WKT FIX: dropped {dropped}]")
        else:
            print(f"\n  -- {tag}")

        y_tr  = y_all.loc[tr_idx]; y_te  = y_all.loc[te_idx]
        naive = naive_all.loc[te_idx]
        mask_tr = ~y_tr.isna(); mask_te = ~y_te.isna()

        print(f"     Features: {len(feat_cols)}   "
              f"NaN drop: train={~mask_tr.sum()}  test={~mask_te.sum()}")
        print(f"     Target -- mean={y_te[mask_te].mean():.3f}  "
              f"std={y_te[mask_te].std():.3f}  "
              f"min={y_te[mask_te].min():.3f}  "
              f"max={y_te[mask_te].max():.3f}")

        X_tr = X_all.loc[tr_idx]; X_te = X_all.loc[te_idx]
        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr); X_te_s = scaler.transform(X_te)

        # Ridge
        rdg = Ridge(alpha=1.0)
        rdg.fit(X_tr_s[mask_tr], y_tr[mask_tr])
        yp_r = rdg.predict(X_te_s[mask_te])
        m_r  = reg_metrics(y_te[mask_te], yp_r, naive[mask_te], naive_label)
        rows.append({"Format": fmt, "Target": tag, "Model": "Ridge", **m_r})
        print(f"    Ridge    R2={m_r['R2']:+.4f}  MAE={m_r['MAE']:.4f}  "
              f"Naive_MAE={m_r['Naive_MAE']:.4f}  Uplift={m_r['Uplift_pct']:+.1f}%")

        # LightGBM
        lgm = lgb.LGBMRegressor(**LGB_PARAMS)
        lgm.fit(X_tr.loc[mask_tr], y_tr[mask_tr],
                eval_set=[(X_te.loc[mask_te], y_te[mask_te])],
                callbacks=[lgb.early_stopping(40, verbose=False), lgb.log_evaluation(-1)])
        yp_l = lgm.predict(X_te.loc[mask_te])
        m_l  = reg_metrics(y_te[mask_te], yp_l, naive[mask_te], naive_label)
        rows.append({"Format": fmt, "Target": tag, "Model": "LightGBM", **m_l})
        print(f"    LightGBM R2={m_l['R2']:+.4f}  MAE={m_l['MAE']:.4f}  "
              f"Naive_MAE={m_l['Naive_MAE']:.4f}  Uplift={m_l['Uplift_pct']:+.1f}%")

        save_shap(lgm, X_te.loc[mask_te], feat_cols, fmt, tag, OUTPUT_DIR)

    return pd.DataFrame(rows)


# ══════════════════════════════════════════════════════════════════════════════
# Summary
# ══════════════════════════════════════════════════════════════════════════════
def print_summary(df):
    bar = "=" * 80
    print(f"\n\n{bar}\n  FULL RESULTS TABLE\n{bar}")
    print(df.to_string(index=False))

    best = df.loc[df.groupby(["Format","Target"])["R2"].idxmax()]
    print(f"\n\n{bar}\n  BEST MODEL PER FORMAT x TARGET\n{bar}")
    print(best[["Format","Target","Model","R2","MAE","Naive_MAE",
                "Uplift_pct","Naive_src"]].to_string(index=False))

    pivot_r2 = best.pivot_table(index="Target", columns="Format", values="R2")
    print(f"\n\n{bar}\n  R2 ACROSS FORMATS (best model per cell)\n{bar}")
    print(pivot_r2.to_string())

    pivot_up = best.pivot_table(index="Target", columns="Format", values="Uplift_pct")
    print(f"\n\n{bar}\n  UPLIFT % OVER NAIVE BASELINE\n{bar}")
    print(pivot_up.to_string())

    print(f"\n{bar}")
    print("  WKT FIX IMPACT NOTE")
    print(f"{bar}")
    wkt_rows = best[best["Target"].str.startswith("WKT")]
    print(wkt_rows[["Format","Target","Model","R2","Uplift_pct"]].to_string(index=False))
    print("\n  Compare WKT R2 values above against v3 (~0.92).")
    print("  Any remaining signal is genuine pre-match prediction.")
    print(f"\n  SHAP plots -> {OUTPUT_DIR}\n")


# ══════════════════════════════════════════════════════════════════════════════
# Main
# ══════════════════════════════════════════════════════════════════════════════
def main():
    t0 = time.time()
    print("BowlerIQ v3 + WKT Fix")
    print(f"Data: {DATA_PATH}\n")

    df = pd.read_parquet(DATA_PATH)
    print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")
    df["match_type"] = df["match_type"].str.strip().str.upper()
    df.loc[df["match_type"] == "T20I", "match_type"] = "T20"
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])
    print(f"match_type distribution:\n{df['match_type'].value_counts().to_string()}\n")

    df = engineer_rolling(df)

    all_results = []
    for fmt in FORMATS:
        df_fmt = df[df["match_type"] == fmt.upper()].copy()
        if len(df_fmt) < 300:
            print(f"\nSkipping {fmt} — only {len(df_fmt)} rows"); continue
        result_df = run_format(fmt, df_fmt)
        result_df = result_df.loc[:, ~result_df.columns.duplicated()]
        all_results.append(result_df)

    if not all_results:
        print("No formats processed."); return

    full = pd.concat(all_results, ignore_index=True)
    out_csv = os.path.join(OUTPUT_DIR, "results_v3_wktfix.csv")
    full.to_csv(out_csv, index=False)
    print(f"\nCSV saved -> {out_csv}")
    print_summary(full)
    print(f"Total runtime: {(time.time()-t0)/60:.1f} min")


if __name__ == "__main__":
    main()

BowlerIQ v3 + WKT Fix
Data: C:\Users\HPP\Bowleriq\abt_with_bpi_rowwise.parquet

Loaded 49,945 rows x 170 columns
match_type distribution:
match_type
T20     22917
ODI     19253
TEST     7775

  Engineering rolling baseline columns...
    _bpi_last5_form            NaN remaining (absolute debuts): 3070
    _bpi_career_avg            NaN remaining (absolute debuts): 3070
    _wkt_last5_form            NaN remaining (absolute debuts): 3070
    _wkt_career_avg            NaN remaining (absolute debuts): 3070
    _dot_last5_form            NaN remaining (absolute debuts): 3070
    _dot_career_avg            NaN remaining (absolute debuts): 3070
    _str_last5_form            NaN remaining (absolute debuts): 3070
    _str_career_avg            NaN remaining (absolute debuts): 3070

  FORMAT: T20   (22,917 rows)
    INFO: dot_pct_last_5_t20 not in data -> career avg used as form baseline
    INFO: strike_rate_last_5_t20 not in data -> career avg used as form baseline
  Train: 18,333   Test: 4

In [2]:
"""
BowlerIQ — Modelling Pipeline v4 (Modelling Part 2)
======================================================
Extends v3_wktfix with a second nonlinear model: XGBoost

Model set for every Format × Target:
  1. Ridge        — linear baseline (L2 regularised)
  2. LightGBM     — gradient boosted trees, leaf-wise splitting,
                    histogram-based (fast, strong on tabular data)
  3. XGBoost      — gradient boosted trees, level-wise splitting,
                    different regularisation (L1+L2+gamma), exact/hist mode
                    Academic rationale: LGB and XGB are peers under the GBDT
                    framework but differ in tree growing strategy, shrinkage,
                    and handling of sparsity. Comparing both strengthens the
                    claim that results are robust to GBDT implementation choice
                    rather than an artifact of one library's defaults.

WKT fix included: wickets_last_5_{fmt} dropped from WKT_DEV feature matrices.
SHAP saved for both LightGBM and XGBoost on every target.
Output directory: modelling_results_v4
"""

import warnings, os, time
warnings.filterwarnings("ignore")

import numpy  as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import shap

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute         import SimpleImputer
from sklearn.linear_model  import Ridge
from sklearn.metrics        import r2_score, mean_absolute_error, mean_squared_error
import lightgbm as lgb
import xgboost  as xgb

# ── Config ────────────────────────────────────────────────────────────────────
DATA_PATH  = r"C:\Users\HPP\Bowleriq\abt_with_bpi_rowwise.parquet"
OUTPUT_DIR = r"C:\Users\HPP\Bowleriq\modelling_results_v4"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FORMATS    = ["T20", "ODI", "Test"]
RAND_STATE = 42

# ── Column taxonomy ───────────────────────────────────────────────────────────
LEAKAGE_OUTCOMES = [
    "balls_bowled","runs_conceded","wickets","dot_balls","total_extras",
    "wides","noballs","byes","legbyes","fours_conceded","sixes_conceded",
    "wickets_bowled","wickets_caught","wickets_lbw","wickets_stumped",
    "wickets_caught_and_bowled","wickets_hit_wicket",
    "balls_powerplay_t20","balls_middle_t20","balls_death_t20",
    "runs_powerplay_t20","runs_middle_t20","runs_death_t20",
    "wickets_powerplay_t20","wickets_middle_t20","wickets_death_t20",
    "dots_powerplay_t20","dots_death_t20",
    "balls_powerplay_odi","balls_middle_odi","balls_death_odi",
    "runs_powerplay_odi","runs_middle_odi","runs_death_odi",
    "wickets_powerplay_odi","wickets_middle_odi","wickets_death_odi",
    "balls_team_spell_1","balls_team_spell_2",
    "runs_team_spell_1","runs_team_spell_2",
    "wickets_team_spell_1","wickets_team_spell_2",
    "first_over","last_over","overs_bowled_distinct",
    "balls_to_rhb","balls_to_lhb",
    "economy_rate","strike_rate","bowling_average",
    "dot_ball_percentage","overs_bowled",
    "boundary_percentage","bowled_percentage","caught_percentage",
    "lbw_percentage","rhb_percentage","lhb_percentage",
    "dots_powerplay_pct_t20","dots_death_pct_t20",
    "winner","result_type","win_by_runs","win_by_wickets","win_method",
    "bowling_team_won",
]

LEAKAGE_BPI = [
    "economy_score","strike_score","wicket_score","dot_score",
    "BPI_Raw","Opposition_Factor","Opposition_Factor_Capped",
    "BPI_OppAdj_Capped","BPI_Final",
]

IDENTIFIERS = [
    "match_id","bowler","bowler_espn_id","bowler_full_name","bowler_dob",
    "venue_id","venue_original","venue","venue_canonical","venue_city",
    "bowling_team","opponent_team",
    "bowling_team_country","bowling_team_code","bowling_team_region",
    "opponent_team_country","opponent_team_code","opponent_team_region",
    "toss_winner","match_type","date",
    "economy_last_5",
    "economy_last_5_t20","economy_last_5_odi","economy_last_5_test",
    "_bpi_last5_form","_bpi_career_avg",
    "_wkt_last5_form","_wkt_career_avg",
    "_dot_last5_form","_dot_career_avg",
    "_str_last5_form","_str_career_avg",
]

ALL_DROP = set(LEAKAGE_OUTCOMES) | set(LEAKAGE_BPI) | set(IDENTIFIERS)

CAT_COLS = [
    "bowler_batting_style","bowler_bowling_style","toss_decision",
    "career_stage","bowling_team_type","bowling_team_member_type",
    "opponent_team_type","opponent_team_member_type",
    "bowler_metadata_quality","venue_country",
]

BOOL_COLS = [
    "is_home_match","bowling_team_won_toss","is_debut","economy_improving",
    "venue_specialist","opponent_specialist","first_time_at_venue",
    "first_time_vs_opponent","opponent_is_full_member","opponent_is_top8",
]

FORMAT_EXCLUSIVE = {
    "T20" : ["career_matches_t20","career_wickets_t20","career_economy_t20",
             "wickets_last_5_t20","economy_last_5_t20"],
    "ODI" : ["career_matches_odi","career_wickets_odi","career_economy_odi",
             "wickets_last_5_odi","economy_last_5_odi"],
    "Test": ["career_matches_test","career_wickets_test","career_economy_test",
             "wickets_last_5_test","economy_last_5_test"],
}

FORMAT_CROSS_DROP = {
    "T20" : set(FORMAT_EXCLUSIVE["ODI"])  | set(FORMAT_EXCLUSIVE["Test"]),
    "ODI" : set(FORMAT_EXCLUSIVE["T20"])  | set(FORMAT_EXCLUSIVE["Test"]),
    "Test": set(FORMAT_EXCLUSIVE["T20"])  | set(FORMAT_EXCLUSIVE["ODI"]),
}

WKT_EXTRA_DROP = {
    "T20" : {"wickets_last_5_t20"},
    "ODI" : {"wickets_last_5_odi"},
    "Test": {"wickets_last_5_test"},
}

# ── Model hyperparameters (untuned defaults — tuning is Script 3) ─────────────
LGB_PARAMS = dict(
    n_estimators=400, learning_rate=0.04, num_leaves=31,
    min_child_samples=30, subsample=0.8, colsample_bytree=0.8,
    random_state=RAND_STATE, n_jobs=-1, verbosity=-1,
)

# XGBoost: hist method matches LGB's histogram approach for fair comparison.
# max_depth=5 ~ num_leaves=31 in terms of tree complexity.
# gamma (min_split_loss) provides additional pruning absent in LGB defaults.
XGB_PARAMS = dict(
    n_estimators=400,
    learning_rate=0.04,
    max_depth=5,
    min_child_weight=30,    # analogous to min_child_samples in LightGBM
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.1,              # min loss reduction for split — XGB-specific pruning
    reg_alpha=0.1,          # L1 regularisation
    reg_lambda=1.0,         # L2 regularisation (XGB default is 1)
    tree_method="hist",     # histogram-based for speed parity with LGB
    random_state=RAND_STATE,
    n_jobs=-1,
    verbosity=0,
    early_stopping_rounds=40,
)


# ══════════════════════════════════════════════════════════════════════════════
# Rolling feature engineering
# ══════════════════════════════════════════════════════════════════════════════
def _rolling_last5_career(df, src_col):
    career = (df.groupby("bowler")[src_col]
                .transform(lambda s: s.shift(1).expanding().mean()))
    last5  = (df.groupby(["bowler", "match_type"])[src_col]
                .transform(lambda s: s.shift(1).rolling(window=5, min_periods=1).mean()))
    return last5.fillna(career), career


def engineer_rolling(df):
    print("  Engineering rolling baseline columns...")
    df = df.sort_values(["bowler", "match_type", "date"]).reset_index(drop=True)
    df["_bpi_last5_form"], df["_bpi_career_avg"] = _rolling_last5_career(df, "BPI_Final")
    df["_wkt_last5_form"], df["_wkt_career_avg"] = _rolling_last5_career(df, "wickets")
    df["_dot_last5_form"], df["_dot_career_avg"] = _rolling_last5_career(df, "dot_ball_percentage")
    df["_str_last5_form"], df["_str_career_avg"] = _rolling_last5_career(df, "strike_rate")
    for col in ["_bpi_last5_form","_bpi_career_avg","_wkt_last5_form","_wkt_career_avg",
                "_dot_last5_form","_dot_career_avg","_str_last5_form","_str_career_avg"]:
        n = df[col].isna().sum()
        if n > 0:
            print(f"    {col:25s}  NaN (absolute debuts): {n}")
    return df


# ══════════════════════════════════════════════════════════════════════════════
# Encoding, feature builder, helpers
# ══════════════════════════════════════════════════════════════════════════════
def encode_and_cast(df):
    for col in CAT_COLS:
        if col in df.columns:
            df[col] = df[col].fillna("__MISSING__").astype(str)
            df[col] = LabelEncoder().fit_transform(df[col])
    for col in BOOL_COLS:
        if col in df.columns:
            df[col] = (df[col].fillna(0)
                              .map(lambda x: 1 if str(x).strip().lower() in ("1","true","yes") else 0)
                              .astype(int))
    return df


def build_features(df, fmt, extra_drop=frozenset()):
    drop_here = ALL_DROP | FORMAT_CROSS_DROP[fmt] | set(extra_drop)
    feat_cols = [c for c in df.columns if c not in drop_here and df[c].dtype != object]
    X = df[feat_cols].copy()
    imp = SimpleImputer(strategy="median")
    return pd.DataFrame(imp.fit_transform(X), columns=feat_cols, index=df.index), feat_cols


def reg_metrics(y_true, y_pred, y_naive, naive_label):
    r2      = r2_score(y_true, y_pred)
    mae     = mean_absolute_error(y_true, y_pred)
    rmse    = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    naive_m = mean_absolute_error(y_true, y_naive)
    uplift  = (naive_m - mae) / naive_m * 100 if naive_m > 0 else 0.0
    return dict(R2=round(r2,4), MAE=round(mae,4), RMSE=round(rmse,4),
                Naive_MAE=round(naive_m,4), Naive_src=naive_label,
                Uplift_pct=round(uplift,2))


def save_shap(model, X_te, feat_cols, fmt, tag, out_dir, prefix):
    try:
        X_s = X_te.sample(min(600, len(X_te)), random_state=RAND_STATE)
        exp = shap.TreeExplainer(model)
        sv  = exp.shap_values(X_s)
        if not isinstance(sv, np.ndarray): sv = np.array(sv)
        if sv.ndim == 3: sv = np.mean(np.abs(sv), axis=0)
        mean_abs = np.abs(sv).mean(axis=0)
        n     = min(len(feat_cols), len(mean_abs))
        cols_ = feat_cols[:n]; vals_ = mean_abs[:n]
        idx   = np.argsort(vals_)[-20:][::-1]
        model_label = "LightGBM" if prefix == "lgb" else "XGBoost"
        fig, ax = plt.subplots(figsize=(9, 6))
        ax.barh([cols_[i] for i in idx[::-1]], [vals_[i] for i in idx[::-1]],
                color="steelblue" if prefix == "lgb" else "darkorange")
        ax.set_xlabel("Mean |SHAP value|")
        ax.set_title(f"Top-20 Features [{model_label}] — {fmt} / {tag}")
        plt.tight_layout()
        fpath = os.path.join(out_dir, f"shap_{prefix}_{fmt}_{tag}.png")
        plt.savefig(fpath, dpi=120); plt.close()
        print(f"        SHAP[{prefix}] -> {fpath}")
    except Exception as e:
        print(f"        SHAP[{prefix}] skipped ({e})")


def make_baseline(primary, fallback_career, global_median):
    return primary.copy().fillna(fallback_career).fillna(global_median)


# ══════════════════════════════════════════════════════════════════════════════
# Per-format pipeline
# ══════════════════════════════════════════════════════════════════════════════
def run_format(fmt, df_fmt):
    print(f"\n{'='*68}")
    print(f"  FORMAT: {fmt}   ({len(df_fmt):,} rows)")
    print(f"{'='*68}")

    fmt_lower = fmt.lower()
    df_fmt    = encode_and_cast(df_fmt.copy())
    wkt_drop  = WKT_EXTRA_DROP.get(fmt, set())

    eco_raw  = df_fmt["economy_rate"].copy()
    bpi_raw  = df_fmt["BPI_Final"].copy()
    wkt_raw  = df_fmt["wickets"].copy()
    dot_raw  = df_fmt["dot_ball_percentage"].copy()
    str_raw  = df_fmt["strike_rate"].copy()

    career_eco = df_fmt["career_economy"].copy()
    career_wkt = df_fmt["career_wickets_per_match"].copy()
    career_dot = df_fmt["career_dot_ball_pct"].copy()
    career_str = df_fmt["career_strike_rate"].copy()

    def _get_col(col_name, career_fallback):
        if col_name in df_fmt.columns: return df_fmt[col_name].copy()
        print(f"    INFO: {col_name} not in data -> career avg used as form baseline")
        return career_fallback.copy()

    med_eco = career_eco.median(); med_wkt = career_wkt.median()
    med_dot = career_dot.median(); med_str = career_str.median()
    med_bpi = df_fmt["_bpi_career_avg"].median()

    eco_bl_form   = make_baseline(_get_col(f"economy_last_5_{fmt_lower}", career_eco),   career_eco,                   med_eco)
    eco_bl_career = career_eco.fillna(med_eco)
    bpi_bl_form   = make_baseline(df_fmt["_bpi_last5_form"],                              df_fmt["_bpi_career_avg"],    med_bpi)
    bpi_bl_career = df_fmt["_bpi_career_avg"].fillna(med_bpi)
    wkt_bl_form   = make_baseline(_get_col(f"wickets_last_5_{fmt_lower}",  career_wkt),  df_fmt["_wkt_career_avg"],    med_wkt)
    wkt_bl_career = df_fmt["_wkt_career_avg"].fillna(med_wkt)
    dot_bl_form   = make_baseline(_get_col(f"dot_pct_last_5_{fmt_lower}",  career_dot),  df_fmt["_dot_career_avg"],    med_dot)
    dot_bl_career = df_fmt["_dot_career_avg"].fillna(med_dot)
    str_bl_form   = make_baseline(_get_col(f"strike_rate_last_5_{fmt_lower}", career_str), df_fmt["_str_career_avg"],  med_str)
    str_bl_career = df_fmt["_str_career_avg"].fillna(med_str)

    TARGETS = {
        "ECO_DEV_form"   : (eco_raw - eco_bl_form,   eco_bl_form,    "last5_eco_fmt",       False),
        "ECO_DEV_career" : (eco_raw - eco_bl_career,  eco_bl_career,  "career_eco",           False),
        "BPI_DEV_form"   : (bpi_raw - bpi_bl_form,   bpi_bl_form,    "last5_BPI_fmt",        False),
        "BPI_DEV_career" : (bpi_raw - bpi_bl_career,  bpi_bl_career,  "career_BPI",           False),
        "WKT_DEV_form"   : (wkt_raw - wkt_bl_form,   wkt_bl_form,    "last5_wkt_fmt",        True),
        "WKT_DEV_career" : (wkt_raw - wkt_bl_career,  wkt_bl_career,  "career_wkt_per_match", True),
        "DOT_DEV_form"   : (dot_raw - dot_bl_form,   dot_bl_form,    "last5_dot_pct_fmt",    False),
        "DOT_DEV_career" : (dot_raw - dot_bl_career,  dot_bl_career,  "career_dot_pct",       False),
        "STR_DEV_form"   : (str_raw - str_bl_form,   str_bl_form,    "last5_str_fmt",        False),
        "STR_DEV_career" : (str_raw - str_bl_career,  str_bl_career,  "career_str",           False),
    }

    sorted_idx = df_fmt["date"].sort_values().index
    n_tr   = int(len(sorted_idx) * 0.80)
    tr_idx = sorted_idx[:n_tr]; te_idx = sorted_idx[n_tr:]
    print(f"  Train: {len(tr_idx):,}   Test: {len(te_idx):,}   "
          f"(cutoff ~ {df_fmt.loc[te_idx,'date'].min().date()})")

    rows = []
    for tag, (y_all, naive_all, naive_label, is_wkt) in TARGETS.items():

        extra   = wkt_drop if is_wkt else frozenset()
        X_all, feat_cols = build_features(df_fmt, fmt, extra_drop=extra)
        wkt_note = f" [WKT-fix: -{wkt_drop & set(df_fmt.columns)}]" if is_wkt else ""
        print(f"\n  -- {tag}{wkt_note}  (naive={naive_label})")

        y_tr  = y_all.loc[tr_idx]; y_te  = y_all.loc[te_idx]
        naive = naive_all.loc[te_idx]
        mask_tr = ~y_tr.isna(); mask_te = ~y_te.isna()
        print(f"     Features: {len(feat_cols)}   "
              f"Target -- mean={y_te[mask_te].mean():.3f}  std={y_te[mask_te].std():.3f}")

        X_tr = X_all.loc[tr_idx]; X_te = X_all.loc[te_idx]
        scaler = StandardScaler()
        X_tr_s = scaler.fit_transform(X_tr); X_te_s = scaler.transform(X_te)

        # ── 1. Ridge ──────────────────────────────────────────────────────────
        rdg = Ridge(alpha=1.0)
        rdg.fit(X_tr_s[mask_tr], y_tr[mask_tr])
        yp_r = rdg.predict(X_te_s[mask_te])
        m_r  = reg_metrics(y_te[mask_te], yp_r, naive[mask_te], naive_label)
        rows.append({"Format":fmt,"Target":tag,"Model":"Ridge",**m_r})
        print(f"    Ridge     R2={m_r['R2']:+.4f}  MAE={m_r['MAE']:.4f}  "
              f"Naive={m_r['Naive_MAE']:.4f}  Uplift={m_r['Uplift_pct']:+.1f}%")

        # ── 2. LightGBM ───────────────────────────────────────────────────────
        lgm = lgb.LGBMRegressor(**LGB_PARAMS)
        lgm.fit(X_tr.loc[mask_tr], y_tr[mask_tr],
                eval_set=[(X_te.loc[mask_te], y_te[mask_te])],
                callbacks=[lgb.early_stopping(40, verbose=False), lgb.log_evaluation(-1)])
        yp_l = lgm.predict(X_te.loc[mask_te])
        m_l  = reg_metrics(y_te[mask_te], yp_l, naive[mask_te], naive_label)
        rows.append({"Format":fmt,"Target":tag,"Model":"LightGBM",**m_l})
        print(f"    LightGBM  R2={m_l['R2']:+.4f}  MAE={m_l['MAE']:.4f}  "
              f"Naive={m_l['Naive_MAE']:.4f}  Uplift={m_l['Uplift_pct']:+.1f}%")
        save_shap(lgm, X_te.loc[mask_te], feat_cols, fmt, tag, OUTPUT_DIR, "lgb")

        # ── 3. XGBoost ────────────────────────────────────────────────────────
        xgm = xgb.XGBRegressor(**XGB_PARAMS)
        xgm.fit(X_tr.loc[mask_tr], y_tr[mask_tr],
                eval_set=[(X_te.loc[mask_te], y_te[mask_te])],
                verbose=False)
        yp_x = xgm.predict(X_te.loc[mask_te])
        m_x  = reg_metrics(y_te[mask_te], yp_x, naive[mask_te], naive_label)
        rows.append({"Format":fmt,"Target":tag,"Model":"XGBoost",**m_x})
        print(f"    XGBoost   R2={m_x['R2']:+.4f}  MAE={m_x['MAE']:.4f}  "
              f"Naive={m_x['Naive_MAE']:.4f}  Uplift={m_x['Uplift_pct']:+.1f}%")
        save_shap(xgm, X_te.loc[mask_te], feat_cols, fmt, tag, OUTPUT_DIR, "xgb")

    return pd.DataFrame(rows)


# ══════════════════════════════════════════════════════════════════════════════
# Summary
# ══════════════════════════════════════════════════════════════════════════════
def print_summary(df):
    bar = "=" * 82
    print(f"\n\n{bar}\n  FULL RESULTS TABLE\n{bar}")
    print(df.to_string(index=False))

    best = df.loc[df.groupby(["Format","Target"])["R2"].idxmax()]
    print(f"\n\n{bar}\n  BEST MODEL PER FORMAT x TARGET\n{bar}")
    print(best[["Format","Target","Model","R2","MAE","Naive_MAE",
                "Uplift_pct","Naive_src"]].to_string(index=False))

    # R2 pivot
    pivot_r2 = best.pivot_table(index="Target", columns="Format", values="R2")
    print(f"\n\n{bar}\n  R2 ACROSS FORMATS (best model per cell)\n{bar}")
    print(pivot_r2.to_string())

    # Model win counts
    print(f"\n\n{bar}\n  MODEL WIN COUNTS (# targets where each model has best R2)\n{bar}")
    win_counts = best.groupby("Model")["Target"].count().sort_values(ascending=False)
    print(win_counts.to_string())

    # Per-target model comparison (Ridge vs LGB vs XGB)
    print(f"\n\n{bar}\n  R2 BY MODEL — GENUINE TARGETS ONLY\n{bar}")
    genuine = ["ECO_DEV_form","ECO_DEV_career","DOT_DEV_form","DOT_DEV_career",
               "STR_DEV_form","STR_DEV_career"]
    df_g  = df[df["Target"].isin(genuine)]
    pivot_model = df_g.pivot_table(index=["Format","Target"], columns="Model", values="R2")
    print(pivot_model.to_string())

    print(f"\n\n{bar}\n  UPLIFT % OVER NAIVE BASELINE (best model per cell)\n{bar}")
    pivot_up = best.pivot_table(index="Target", columns="Format", values="Uplift_pct")
    print(pivot_up.to_string())

    print(f"""
{bar}
  INTERPRETATION: LightGBM vs XGBoost
{bar}
  If LightGBM and XGBoost produce similar R2 values across targets
  and formats, this strengthens the robustness claim:
    "Results are not an artifact of one GBDT library's defaults."
  
  If one consistently outperforms the other, tuning (Script 3) may
  reveal whether the gap closes with optimal hyperparameters or
  reflects a structural advantage for this dataset.

  WKT_DEV targets: R2 values post-fix are the genuine predictive
  signal. Expect much lower than v3's artifactual 0.92.

  SHAP plots: {OUTPUT_DIR}
""")


# ══════════════════════════════════════════════════════════════════════════════
# Main
# ══════════════════════════════════════════════════════════════════════════════
def main():
    t0 = time.time()
    print("BowlerIQ Modelling v4 — Ridge + LightGBM + XGBoost")
    print(f"Data: {DATA_PATH}\n")

    df = pd.read_parquet(DATA_PATH)
    print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")
    df["match_type"] = df["match_type"].str.strip().str.upper()
    df.loc[df["match_type"] == "T20I", "match_type"] = "T20"
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])
    print(f"match_type distribution:\n{df['match_type'].value_counts().to_string()}\n")

    df = engineer_rolling(df)

    all_results = []
    for fmt in FORMATS:
        df_fmt = df[df["match_type"] == fmt.upper()].copy()
        if len(df_fmt) < 300:
            print(f"\nSkipping {fmt} — only {len(df_fmt)} rows"); continue
        result_df = run_format(fmt, df_fmt)
        result_df = result_df.loc[:, ~result_df.columns.duplicated()]
        all_results.append(result_df)

    if not all_results:
        print("No formats processed."); return

    full = pd.concat(all_results, ignore_index=True)
    out_csv = os.path.join(OUTPUT_DIR, "results_v4_lgb_xgb.csv")
    full.to_csv(out_csv, index=False)
    print(f"\nCSV saved -> {out_csv}")
    print_summary(full)
    print(f"Total runtime: {(time.time()-t0)/60:.1f} min")


if __name__ == "__main__":
    main()

BowlerIQ Modelling v4 — Ridge + LightGBM + XGBoost
Data: C:\Users\HPP\Bowleriq\abt_with_bpi_rowwise.parquet

Loaded 49,945 rows x 170 columns
match_type distribution:
match_type
T20     22917
ODI     19253
TEST     7775

  Engineering rolling baseline columns...
    _bpi_last5_form            NaN (absolute debuts): 3070
    _bpi_career_avg            NaN (absolute debuts): 3070
    _wkt_last5_form            NaN (absolute debuts): 3070
    _wkt_career_avg            NaN (absolute debuts): 3070
    _dot_last5_form            NaN (absolute debuts): 3070
    _dot_career_avg            NaN (absolute debuts): 3070
    _str_last5_form            NaN (absolute debuts): 3070
    _str_career_avg            NaN (absolute debuts): 3070

  FORMAT: T20   (22,917 rows)
    INFO: dot_pct_last_5_t20 not in data -> career avg used as form baseline
    INFO: strike_rate_last_5_t20 not in data -> career avg used as form baseline
  Train: 18,333   Test: 4,584   (cutoff ~ 2024-11-26)

  -- ECO_DEV_form  (n

In [3]:
"""
BowlerIQ — Hyperparameter Tuning (Script 3)
============================================
Optuna-based tuning for LightGBM and XGBoost on the top genuine targets.

Temporal split design (3-way, no test set contamination):
  ┌────────────────────────────────────────────────────────────┐
  │  70% train-for-tuning │ 10% validation │ 20% held-out test │
  └────────────────────────────────────────────────────────────┘
  - Optuna maximises R2 on the VALIDATION set (10%)
  - Final model refits on train+val (80%) with best params
  - Final R2 reported on held-out TEST set (20%) — never seen during tuning

Targets tuned (genuine results only; WKT/BPI excluded):
  ECO_DEV_form, ECO_DEV_career
  DOT_DEV_form, DOT_DEV_career
  STR_DEV_form, STR_DEV_career

Models tuned: LightGBM, XGBoost
Formats: T20, ODI, Test

Output:
  tuning_results.csv        — full comparison: default vs tuned, both models
  best_params_lgb.csv       — best Optuna params per format × target for LGB
  best_params_xgb.csv       — best Optuna params per format × target for XGB
  improvement_summary.csv   — R2 delta: tuned vs default, sorted by gain

Runtime estimate: ~20–40 min depending on N_TRIALS and hardware.
Reduce N_TRIALS to 30 for a quick sanity check.
"""

import warnings, os, time
warnings.filterwarnings("ignore")

import numpy  as np
import pandas as pd
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute         import SimpleImputer
from sklearn.metrics        import r2_score, mean_absolute_error, mean_squared_error
import lightgbm as lgb
import xgboost  as xgb

# ── Config ────────────────────────────────────────────────────────────────────
DATA_PATH  = r"C:\Users\HPP\Bowleriq\abt_with_bpi_rowwise.parquet"
OUTPUT_DIR = r"C:\Users\HPP\Bowleriq\modelling_results_tuned"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FORMATS    = ["T20", "ODI", "Test"]
RAND_STATE = 42

# Targets to tune — genuine results only, WKT and BPI excluded.
# WKT was an artifact; BPI R2 is structurally low due to its wicket weighting.
# Tuning won't rescue fundamentally noisy targets.
TUNE_TARGETS = [
    "ECO_DEV_form",
    "ECO_DEV_career",
    "DOT_DEV_form",
    "DOT_DEV_career",
    "STR_DEV_form",
    "STR_DEV_career",
]

# Number of Optuna trials per model per (format, target) combination.
# 50 is a reasonable sweep. Reduce to 30 for faster results.
N_TRIALS = 50

# ── Column taxonomy (identical to v4) ─────────────────────────────────────────
LEAKAGE_OUTCOMES = [
    "balls_bowled","runs_conceded","wickets","dot_balls","total_extras",
    "wides","noballs","byes","legbyes","fours_conceded","sixes_conceded",
    "wickets_bowled","wickets_caught","wickets_lbw","wickets_stumped",
    "wickets_caught_and_bowled","wickets_hit_wicket",
    "balls_powerplay_t20","balls_middle_t20","balls_death_t20",
    "runs_powerplay_t20","runs_middle_t20","runs_death_t20",
    "wickets_powerplay_t20","wickets_middle_t20","wickets_death_t20",
    "dots_powerplay_t20","dots_death_t20",
    "balls_powerplay_odi","balls_middle_odi","balls_death_odi",
    "runs_powerplay_odi","runs_middle_odi","runs_death_odi",
    "wickets_powerplay_odi","wickets_middle_odi","wickets_death_odi",
    "balls_team_spell_1","balls_team_spell_2",
    "runs_team_spell_1","runs_team_spell_2",
    "wickets_team_spell_1","wickets_team_spell_2",
    "first_over","last_over","overs_bowled_distinct",
    "balls_to_rhb","balls_to_lhb",
    "economy_rate","strike_rate","bowling_average",
    "dot_ball_percentage","overs_bowled",
    "boundary_percentage","bowled_percentage","caught_percentage",
    "lbw_percentage","rhb_percentage","lhb_percentage",
    "dots_powerplay_pct_t20","dots_death_pct_t20",
    "winner","result_type","win_by_runs","win_by_wickets","win_method",
    "bowling_team_won",
]

LEAKAGE_BPI = [
    "economy_score","strike_score","wicket_score","dot_score",
    "BPI_Raw","Opposition_Factor","Opposition_Factor_Capped",
    "BPI_OppAdj_Capped","BPI_Final",
]

IDENTIFIERS = [
    "match_id","bowler","bowler_espn_id","bowler_full_name","bowler_dob",
    "venue_id","venue_original","venue","venue_canonical","venue_city",
    "bowling_team","opponent_team",
    "bowling_team_country","bowling_team_code","bowling_team_region",
    "opponent_team_country","opponent_team_code","opponent_team_region",
    "toss_winner","match_type","date",
    "economy_last_5",
    "economy_last_5_t20","economy_last_5_odi","economy_last_5_test",
    "_bpi_last5_form","_bpi_career_avg",
    "_wkt_last5_form","_wkt_career_avg",
    "_dot_last5_form","_dot_career_avg",
    "_str_last5_form","_str_career_avg",
]

ALL_DROP = set(LEAKAGE_OUTCOMES) | set(LEAKAGE_BPI) | set(IDENTIFIERS)

CAT_COLS = [
    "bowler_batting_style","bowler_bowling_style","toss_decision",
    "career_stage","bowling_team_type","bowling_team_member_type",
    "opponent_team_type","opponent_team_member_type",
    "bowler_metadata_quality","venue_country",
]

BOOL_COLS = [
    "is_home_match","bowling_team_won_toss","is_debut","economy_improving",
    "venue_specialist","opponent_specialist","first_time_at_venue",
    "first_time_vs_opponent","opponent_is_full_member","opponent_is_top8",
]

FORMAT_EXCLUSIVE = {
    "T20" : ["career_matches_t20","career_wickets_t20","career_economy_t20",
             "wickets_last_5_t20","economy_last_5_t20"],
    "ODI" : ["career_matches_odi","career_wickets_odi","career_economy_odi",
             "wickets_last_5_odi","economy_last_5_odi"],
    "Test": ["career_matches_test","career_wickets_test","career_economy_test",
             "wickets_last_5_test","economy_last_5_test"],
}

FORMAT_CROSS_DROP = {
    "T20" : set(FORMAT_EXCLUSIVE["ODI"])  | set(FORMAT_EXCLUSIVE["Test"]),
    "ODI" : set(FORMAT_EXCLUSIVE["T20"])  | set(FORMAT_EXCLUSIVE["Test"]),
    "Test": set(FORMAT_EXCLUSIVE["T20"])  | set(FORMAT_EXCLUSIVE["ODI"]),
}

# Default params (mirrors v4, used as comparison baseline)
LGB_DEFAULTS = dict(
    n_estimators=400, learning_rate=0.04, num_leaves=31,
    min_child_samples=30, subsample=0.8, colsample_bytree=0.8,
    random_state=RAND_STATE, n_jobs=-1, verbosity=-1,
)

XGB_DEFAULTS = dict(
    n_estimators=400, learning_rate=0.04, max_depth=5,
    min_child_weight=30, subsample=0.8, colsample_bytree=0.8,
    gamma=0.1, reg_alpha=0.1, reg_lambda=1.0,
    tree_method="hist", random_state=RAND_STATE, n_jobs=-1, verbosity=0,
    early_stopping_rounds=40,
)


# ══════════════════════════════════════════════════════════════════════════════
# Data loading and preparation (same as v4)
# ══════════════════════════════════════════════════════════════════════════════
def _rolling_last5_career(df, src_col):
    career = (df.groupby("bowler")[src_col]
                .transform(lambda s: s.shift(1).expanding().mean()))
    last5  = (df.groupby(["bowler", "match_type"])[src_col]
                .transform(lambda s: s.shift(1).rolling(window=5, min_periods=1).mean()))
    return last5.fillna(career), career


def engineer_rolling(df):
    print("  Engineering rolling baseline columns...")
    df = df.sort_values(["bowler", "match_type", "date"]).reset_index(drop=True)
    df["_bpi_last5_form"], df["_bpi_career_avg"] = _rolling_last5_career(df, "BPI_Final")
    df["_wkt_last5_form"], df["_wkt_career_avg"] = _rolling_last5_career(df, "wickets")
    df["_dot_last5_form"], df["_dot_career_avg"] = _rolling_last5_career(df, "dot_ball_percentage")
    df["_str_last5_form"], df["_str_career_avg"] = _rolling_last5_career(df, "strike_rate")
    return df


def encode_and_cast(df):
    for col in CAT_COLS:
        if col in df.columns:
            df[col] = df[col].fillna("__MISSING__").astype(str)
            df[col] = LabelEncoder().fit_transform(df[col])
    for col in BOOL_COLS:
        if col in df.columns:
            df[col] = (df[col].fillna(0)
                              .map(lambda x: 1 if str(x).strip().lower() in ("1","true","yes") else 0)
                              .astype(int))
    return df


def build_features(df, fmt):
    drop_here = ALL_DROP | FORMAT_CROSS_DROP[fmt]
    feat_cols = [c for c in df.columns if c not in drop_here and df[c].dtype != object]
    X = df[feat_cols].copy()
    imp = SimpleImputer(strategy="median")
    return pd.DataFrame(imp.fit_transform(X), columns=feat_cols, index=df.index), feat_cols


def make_baseline(primary, fallback_career, global_median):
    return primary.copy().fillna(fallback_career).fillna(global_median)


def reg_metrics(y_true, y_pred, y_naive=None):
    r2   = r2_score(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    out  = dict(R2=round(r2,4), MAE=round(mae,4), RMSE=round(rmse,4))
    if y_naive is not None:
        naive_m = mean_absolute_error(y_true, y_naive)
        uplift  = (naive_m - mae) / naive_m * 100 if naive_m > 0 else 0.0
        out.update(Naive_MAE=round(naive_m,4), Uplift_pct=round(uplift,2))
    return out


def build_all_targets(df_fmt, fmt):
    """Build all 10 deviation target series for a format slice."""
    fmt_lower  = fmt.lower()
    career_eco = df_fmt["career_economy"].copy()
    career_wkt = df_fmt["career_wickets_per_match"].copy()
    career_dot = df_fmt["career_dot_ball_pct"].copy()
    career_str = df_fmt["career_strike_rate"].copy()

    def _get(col, fb):
        return df_fmt[col].copy() if col in df_fmt.columns else fb.copy()

    med_eco = career_eco.median(); med_wkt = career_wkt.median()
    med_dot = career_dot.median(); med_str = career_str.median()
    med_bpi = df_fmt["_bpi_career_avg"].median()

    eco_raw = df_fmt["economy_rate"]; bpi_raw = df_fmt["BPI_Final"]
    wkt_raw = df_fmt["wickets"];      dot_raw = df_fmt["dot_ball_percentage"]
    str_raw = df_fmt["strike_rate"]

    eco_bl_form   = make_baseline(_get(f"economy_last_5_{fmt_lower}",   career_eco), career_eco,               med_eco)
    eco_bl_career = career_eco.fillna(med_eco)
    bpi_bl_form   = make_baseline(df_fmt["_bpi_last5_form"],             df_fmt["_bpi_career_avg"],             med_bpi)
    bpi_bl_career = df_fmt["_bpi_career_avg"].fillna(med_bpi)
    wkt_bl_form   = make_baseline(_get(f"wickets_last_5_{fmt_lower}",   career_wkt), df_fmt["_wkt_career_avg"], med_wkt)
    wkt_bl_career = df_fmt["_wkt_career_avg"].fillna(med_wkt)
    dot_bl_form   = make_baseline(_get(f"dot_pct_last_5_{fmt_lower}",   career_dot), df_fmt["_dot_career_avg"], med_dot)
    dot_bl_career = df_fmt["_dot_career_avg"].fillna(med_dot)
    str_bl_form   = make_baseline(_get(f"strike_rate_last_5_{fmt_lower}", career_str), df_fmt["_str_career_avg"], med_str)
    str_bl_career = df_fmt["_str_career_avg"].fillna(med_str)

    return {
        "ECO_DEV_form"   : (eco_raw - eco_bl_form,   eco_bl_form),
        "ECO_DEV_career" : (eco_raw - eco_bl_career,  eco_bl_career),
        "BPI_DEV_form"   : (bpi_raw - bpi_bl_form,   bpi_bl_form),
        "BPI_DEV_career" : (bpi_raw - bpi_bl_career,  bpi_bl_career),
        "WKT_DEV_form"   : (wkt_raw - wkt_bl_form,   wkt_bl_form),
        "WKT_DEV_career" : (wkt_raw - wkt_bl_career,  wkt_bl_career),
        "DOT_DEV_form"   : (dot_raw - dot_bl_form,   dot_bl_form),
        "DOT_DEV_career" : (dot_raw - dot_bl_career,  dot_bl_career),
        "STR_DEV_form"   : (str_raw - str_bl_form,   str_bl_form),
        "STR_DEV_career" : (str_raw - str_bl_career,  str_bl_career),
    }


# ══════════════════════════════════════════════════════════════════════════════
# Optuna objective functions
# ══════════════════════════════════════════════════════════════════════════════
def lgb_objective(trial, X_tr, y_tr, X_val, y_val):
    """
    Optuna trial for LightGBM. n_estimators is a search param with
    early stopping set high (200) so the trial-level bound matters.
    """
    params = dict(
        n_estimators       = trial.suggest_int("n_estimators", 200, 1500),
        learning_rate      = trial.suggest_float("learning_rate", 0.005, 0.15, log=True),
        num_leaves         = trial.suggest_int("num_leaves", 15, 200),
        min_child_samples  = trial.suggest_int("min_child_samples", 5, 100),
        subsample          = trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree   = trial.suggest_float("colsample_bytree", 0.4, 1.0),
        reg_alpha          = trial.suggest_float("reg_alpha", 1e-5, 5.0, log=True),
        reg_lambda         = trial.suggest_float("reg_lambda", 1e-5, 5.0, log=True),
        min_split_gain     = trial.suggest_float("min_split_gain", 0.0, 1.0),
        random_state       = RAND_STATE,
        n_jobs             = -1,
        verbosity          = -1,
    )
    model = lgb.LGBMRegressor(**params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)],
    )
    return r2_score(y_val, model.predict(X_val))


def xgb_objective(trial, X_tr, y_tr, X_val, y_val):
    """
    Optuna trial for XGBoost.
    """
    params = dict(
        n_estimators       = trial.suggest_int("n_estimators", 200, 1500),
        learning_rate      = trial.suggest_float("learning_rate", 0.005, 0.15, log=True),
        max_depth          = trial.suggest_int("max_depth", 3, 9),
        min_child_weight   = trial.suggest_int("min_child_weight", 5, 100, log=True),
        subsample          = trial.suggest_float("subsample", 0.5, 1.0),
        colsample_bytree   = trial.suggest_float("colsample_bytree", 0.4, 1.0),
        gamma              = trial.suggest_float("gamma", 0.0, 5.0),
        reg_alpha          = trial.suggest_float("reg_alpha", 1e-5, 5.0, log=True),
        reg_lambda         = trial.suggest_float("reg_lambda", 1e-5, 5.0, log=True),
        tree_method        = "hist",
        random_state       = RAND_STATE,
        n_jobs             = -1,
        verbosity          = 0,
        early_stopping_rounds = 50,
    )
    model = xgb.XGBRegressor(**params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    return r2_score(y_val, model.predict(X_val))


# ══════════════════════════════════════════════════════════════════════════════
# Refit with best params on train+val, evaluate on held-out test
# ══════════════════════════════════════════════════════════════════════════════
def refit_lgb(best_params, X_trainval, y_trainval, X_test, y_test, y_naive):
    """Retrain LGB on full 80% (train+val) using best Optuna params."""
    p = {**best_params, "random_state": RAND_STATE, "n_jobs": -1, "verbosity": -1}
    model = lgb.LGBMRegressor(**p)
    model.fit(X_trainval, y_trainval)
    return model, reg_metrics(y_test, model.predict(X_test), y_naive)


def refit_xgb(best_params, X_trainval, y_trainval, X_test, y_test, y_naive):
    """Retrain XGB on full 80% (train+val) using best Optuna params."""
    # Remove early_stopping_rounds — no eval_set during final refit
    p = {k: v for k, v in best_params.items() if k != "early_stopping_rounds"}
    p.update(tree_method="hist", random_state=RAND_STATE, n_jobs=-1, verbosity=0)
    model = xgb.XGBRegressor(**p)
    model.fit(X_trainval, y_trainval)
    return model, reg_metrics(y_test, model.predict(X_test), y_naive)


# ══════════════════════════════════════════════════════════════════════════════
# Default model evaluation (mirrors v4 — for direct comparison)
# ══════════════════════════════════════════════════════════════════════════════
def eval_defaults(X_trainval, y_trainval, X_test, y_test, y_naive):
    """Evaluate default LGB and XGB on the same 80/20 split for comparison."""
    # LGB default
    lgm = lgb.LGBMRegressor(**LGB_DEFAULTS)
    lgm.fit(X_trainval, y_trainval,
            eval_set=[(X_test, y_test)],
            callbacks=[lgb.early_stopping(40, verbose=False), lgb.log_evaluation(-1)])
    m_lgb_def = reg_metrics(y_test, lgm.predict(X_test), y_naive)

    # XGB default
    xgm = xgb.XGBRegressor(**XGB_DEFAULTS)
    xgm.fit(X_trainval, y_trainval, eval_set=[(X_test, y_test)], verbose=False)
    m_xgb_def = reg_metrics(y_test, xgm.predict(X_test), y_naive)

    return m_lgb_def, m_xgb_def


# ══════════════════════════════════════════════════════════════════════════════
# Per-format × target tuning run
# ══════════════════════════════════════════════════════════════════════════════
def tune_target(fmt, tag, df_fmt, targets_dict, feat_cols, X_all):
    """
    Run Optuna tuning for one (format, target) combination.
    Returns a dict of results and best params for both models.
    """
    y_all, naive_all = targets_dict[tag]

    # ── 3-way temporal split: 70 / 10 / 20 ─────────────────────────────────
    sorted_idx   = df_fmt["date"].sort_values().index
    n_total      = len(sorted_idx)
    n_trainval   = int(n_total * 0.80)  # 80% train+val (for final refit)
    n_train      = int(n_total * 0.70)  # 70% for tuning train
    # 10% val = indices [70%..80%)
    # 20% test = indices [80%..100%)

    tr_idx  = sorted_idx[:n_train]
    val_idx = sorted_idx[n_train:n_trainval]
    te_idx  = sorted_idx[n_trainval:]
    tv_idx  = sorted_idx[:n_trainval]   # train+val combined for final refit

    y_tr    = y_all.loc[tr_idx];  y_val  = y_all.loc[val_idx]
    y_te    = y_all.loc[te_idx];  y_tv   = y_all.loc[tv_idx]
    naive   = naive_all.loc[te_idx]

    # Drop NaN rows — align masks
    mask_tr  = ~y_tr.isna();  mask_val = ~y_val.isna()
    mask_te  = ~y_te.isna();  mask_tv  = ~y_tv.isna()

    if mask_te.sum() < 50:
        print(f"    SKIP {fmt}/{tag}: insufficient test rows ({mask_te.sum()})")
        return None

    X_tr  = X_all.loc[tr_idx][mask_tr];   y_tr  = y_tr[mask_tr]
    X_val = X_all.loc[val_idx][mask_val];  y_val = y_val[mask_val]
    X_te  = X_all.loc[te_idx][mask_te];   y_te  = y_te[mask_te]
    X_tv  = X_all.loc[tv_idx][mask_tv];   y_tv  = y_tv[mask_tv]
    naive = naive[mask_te]

    print(f"\n  [{fmt}] {tag}  |  "
          f"train={len(X_tr):,}  val={len(X_val):,}  test={len(X_te):,}")

    # ── Default model evaluation (80% trainval → 20% test) ─────────────────
    m_lgb_def, m_xgb_def = eval_defaults(X_tv, y_tv, X_te, y_te, naive)
    print(f"    Default LGB  R2={m_lgb_def['R2']:+.4f}  "
          f"XGB  R2={m_xgb_def['R2']:+.4f}")

    # ── LightGBM Optuna study ────────────────────────────────────────────────
    print(f"    Tuning LightGBM ({N_TRIALS} trials)...")
    t_lgb = time.time()
    study_lgb = optuna.create_study(direction="maximize",
                                    sampler=optuna.samplers.TPESampler(seed=RAND_STATE))
    study_lgb.optimize(
        lambda trial: lgb_objective(trial, X_tr, y_tr, X_val, y_val),
        n_trials=N_TRIALS, show_progress_bar=False,
    )
    best_params_lgb = study_lgb.best_params
    val_r2_lgb      = study_lgb.best_value
    _, m_lgb_tuned  = refit_lgb(best_params_lgb, X_tv, y_tv, X_te, y_te, naive)
    print(f"    LGB tuned  val_R2={val_r2_lgb:+.4f}  "
          f"test_R2={m_lgb_tuned['R2']:+.4f}  "
          f"gain={m_lgb_tuned['R2']-m_lgb_def['R2']:+.4f}  "
          f"({time.time()-t_lgb:.0f}s)")

    # ── XGBoost Optuna study ─────────────────────────────────────────────────
    print(f"    Tuning XGBoost  ({N_TRIALS} trials)...")
    t_xgb = time.time()
    study_xgb = optuna.create_study(direction="maximize",
                                    sampler=optuna.samplers.TPESampler(seed=RAND_STATE))
    study_xgb.optimize(
        lambda trial: xgb_objective(trial, X_tr, y_tr, X_val, y_val),
        n_trials=N_TRIALS, show_progress_bar=False,
    )
    best_params_xgb = study_xgb.best_params
    val_r2_xgb      = study_xgb.best_value
    _, m_xgb_tuned  = refit_xgb(best_params_xgb, X_tv, y_tv, X_te, y_te, naive)
    print(f"    XGB tuned  val_R2={val_r2_xgb:+.4f}  "
          f"test_R2={m_xgb_tuned['R2']:+.4f}  "
          f"gain={m_xgb_tuned['R2']-m_xgb_def['R2']:+.4f}  "
          f"({time.time()-t_xgb:.0f}s)")

    # ── Assemble result record ────────────────────────────────────────────────
    record = {
        "Format"             : fmt,
        "Target"             : tag,
        "Train_rows"         : len(X_tr),
        "Val_rows"           : len(X_val),
        "Test_rows"          : len(X_te),
        # Default R2
        "LGB_default_R2"     : m_lgb_def["R2"],
        "XGB_default_R2"     : m_xgb_def["R2"],
        # Tuned test R2
        "LGB_tuned_R2"       : m_lgb_tuned["R2"],
        "XGB_tuned_R2"       : m_xgb_tuned["R2"],
        # Tuned val R2 (what Optuna saw)
        "LGB_val_R2"         : round(val_r2_lgb, 4),
        "XGB_val_R2"         : round(val_r2_xgb, 4),
        # MAE
        "LGB_tuned_MAE"      : m_lgb_tuned["MAE"],
        "XGB_tuned_MAE"      : m_xgb_tuned["MAE"],
        # Uplift
        "LGB_tuned_Uplift"   : m_lgb_tuned.get("Uplift_pct", None),
        "XGB_tuned_Uplift"   : m_xgb_tuned.get("Uplift_pct", None),
        # Gain over default
        "LGB_R2_gain"        : round(m_lgb_tuned["R2"] - m_lgb_def["R2"], 4),
        "XGB_R2_gain"        : round(m_xgb_tuned["R2"] - m_xgb_def["R2"], 4),
        # Best tuned model winner
        "Best_tuned_model"   : ("LightGBM" if m_lgb_tuned["R2"] >= m_xgb_tuned["R2"]
                                else "XGBoost"),
        "Best_tuned_R2"      : max(m_lgb_tuned["R2"], m_xgb_tuned["R2"]),
    }

    return record, best_params_lgb, best_params_xgb


# ══════════════════════════════════════════════════════════════════════════════
# Summary printer
# ══════════════════════════════════════════════════════════════════════════════
def print_summary(df):
    bar = "=" * 82
    print(f"\n\n{bar}\n  TUNING RESULTS — DEFAULT vs TUNED R2\n{bar}")
    cols_show = ["Format","Target",
                 "LGB_default_R2","LGB_tuned_R2","LGB_R2_gain",
                 "XGB_default_R2","XGB_tuned_R2","XGB_R2_gain",
                 "Best_tuned_model","Best_tuned_R2"]
    print(df[cols_show].to_string(index=False))

    print(f"\n\n{bar}\n  BEST TUNED R2 PIVOT (Format x Target)\n{bar}")
    pivot = df.pivot_table(index="Target", columns="Format", values="Best_tuned_R2")
    print(pivot.to_string())

    print(f"\n\n{bar}\n  AVERAGE R2 GAIN FROM TUNING\n{bar}")
    print(f"  LightGBM avg gain: {df['LGB_R2_gain'].mean():+.4f}")
    print(f"  XGBoost  avg gain: {df['XGB_R2_gain'].mean():+.4f}")
    print(f"  LightGBM wins:     {(df['Best_tuned_model']=='LightGBM').sum()} / {len(df)}")
    print(f"  XGBoost  wins:     {(df['Best_tuned_model']=='XGBoost').sum()} / {len(df)}")

    print(f"\n\n{bar}\n  TOP GAINS (sorted by LGB R2 gain)\n{bar}")
    top = df.sort_values("LGB_R2_gain", ascending=False)[
        ["Format","Target","LGB_default_R2","LGB_tuned_R2","LGB_R2_gain"]
    ]
    print(top.to_string(index=False))

    print(f"""
{bar}
  PAPER FRAMING NOTE
{bar}
  Report the following three-row comparison table per target in the paper:
    Model          | Default R2 | Tuned R2 | Gain
    LightGBM       |   x.xxxx   |  x.xxxx  | +x.xxxx
    XGBoost        |   x.xxxx   |  x.xxxx  | +x.xxxx

  If tuning gain > 0.03 on top targets: mention in Results as
    "Hyperparameter optimisation yielded an additional +x.xx R2
     improvement for ECO_DEV_career in Test cricket."

  If gain < 0.03: mention in Limitations as
    "Default gradient boosting parameters were near-optimal for
     this dataset scale; tuning provided marginal improvement."

  Outputs: {OUTPUT_DIR}
""")


# ══════════════════════════════════════════════════════════════════════════════
# Main
# ══════════════════════════════════════════════════════════════════════════════
def main():
    t0 = time.time()
    print(f"BowlerIQ Tuning — {N_TRIALS} Optuna trials per model per (format, target)")
    print(f"Data: {DATA_PATH}\n")

    # ── Load and prepare ─────────────────────────────────────────────────────
    df = pd.read_parquet(DATA_PATH)
    print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")
    df["match_type"] = df["match_type"].str.strip().str.upper()
    df.loc[df["match_type"] == "T20I", "match_type"] = "T20"
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df = df.dropna(subset=["date"])

    df = engineer_rolling(df)

    all_records   = []
    all_params_lgb = []
    all_params_xgb = []

    for fmt in FORMATS:
        df_fmt = df[df["match_type"] == fmt.upper()].copy()
        if len(df_fmt) < 300:
            print(f"\nSkipping {fmt} — only {len(df_fmt)} rows"); continue

        df_fmt = encode_and_cast(df_fmt)
        X_all, feat_cols = build_features(df_fmt, fmt)
        targets_dict = build_all_targets(df_fmt, fmt)

        print(f"\n{'='*68}")
        print(f"  FORMAT: {fmt}   ({len(df_fmt):,} rows)   Features: {len(feat_cols)}")
        print(f"{'='*68}")

        for tag in TUNE_TARGETS:
            if tag not in targets_dict:
                print(f"  SKIP: {tag} not in targets_dict"); continue

            result = tune_target(fmt, tag, df_fmt, targets_dict, feat_cols, X_all)
            if result is None: continue

            record, bp_lgb, bp_xgb = result
            all_records.append(record)
            all_params_lgb.append({"Format": fmt, "Target": tag, **bp_lgb})
            all_params_xgb.append({"Format": fmt, "Target": tag, **bp_xgb})

    if not all_records:
        print("No tuning results produced."); return

    results_df  = pd.DataFrame(all_records)
    params_lgb  = pd.DataFrame(all_params_lgb)
    params_xgb  = pd.DataFrame(all_params_xgb)

    results_df.to_csv(os.path.join(OUTPUT_DIR, "tuning_results.csv"),  index=False)
    params_lgb.to_csv(os.path.join(OUTPUT_DIR, "best_params_lgb.csv"), index=False)
    params_xgb.to_csv(os.path.join(OUTPUT_DIR, "best_params_xgb.csv"), index=False)

    # Improvement summary sorted by best R2 gain across both models
    results_df["Max_R2_gain"] = results_df[["LGB_R2_gain","XGB_R2_gain"]].max(axis=1)
    improv = results_df[["Format","Target","LGB_default_R2","LGB_tuned_R2",
                          "XGB_default_R2","XGB_tuned_R2","Max_R2_gain",
                          "Best_tuned_model","Best_tuned_R2"]]
    improv.sort_values("Max_R2_gain", ascending=False).to_csv(
        os.path.join(OUTPUT_DIR, "improvement_summary.csv"), index=False)

    print_summary(results_df)
    print(f"Total tuning runtime: {(time.time()-t0)/60:.1f} min")


if __name__ == "__main__":
    main()

BowlerIQ Tuning — 50 Optuna trials per model per (format, target)
Data: C:\Users\HPP\Bowleriq\abt_with_bpi_rowwise.parquet

Loaded 49,945 rows x 170 columns
  Engineering rolling baseline columns...

  FORMAT: T20   (22,917 rows)   Features: 61

  [T20] ECO_DEV_form  |  train=16,041  val=2,292  test=4,584
    Default LGB  R2=+0.2567  XGB  R2=+0.2570
    Tuning LightGBM (50 trials)...
    LGB tuned  val_R2=+0.3092  test_R2=+0.2496  gain=-0.0071  (223s)
    Tuning XGBoost  (50 trials)...
    XGB tuned  val_R2=+0.3169  test_R2=+0.2266  gain=-0.0304  (306s)

  [T20] ECO_DEV_career  |  train=16,041  val=2,292  test=4,584
    Default LGB  R2=+0.2988  XGB  R2=+0.2959
    Tuning LightGBM (50 trials)...
    LGB tuned  val_R2=+0.3538  test_R2=+0.2732  gain=-0.0256  (242s)
    Tuning XGBoost  (50 trials)...
    XGB tuned  val_R2=+0.3534  test_R2=+0.2760  gain=-0.0199  (220s)

  [T20] DOT_DEV_form  |  train=16,041  val=2,292  test=4,584
    Default LGB  R2=+0.2794  XGB  R2=+0.2824
    Tuning Light